# Breast cancer stage classification: corrected analysis and paper generator

**How to use:** Runtime → Run all. When asked, upload `denseDataOnlyDownload-3.tsv` (the UCSC Xena file you used before).
The last cell downloads `paper.pdf`, `paper.docx` (editable, for journal submission) and `results.zip` (all tables and 300-dpi figures).

Every number in the paper is computed by this notebook; nothing is typed in by hand.

**What changed from the earlier notebook**
1. Labels are early = AJCC Stage I–II and late = Stage III–IV. The old final model used "Stage I vs everything else", which also counted Stage X, "[Discrepancy]" and blank stages as late.
2. Only primary-tumour samples (barcode type 01), one per patient. Normal-tissue (11) and metastatic (06) samples are removed.
3. The data are split before any imputation, scaling or SMOTE. SMOTE runs only inside the training folds of cross-validation.
4. Hyper-parameters are tuned by stratified 5-fold CV. The decision threshold is chosen on out-of-fold predictions (Youden's J), not left at 0.5.
5. The metrics are ROC-AUC with a 95% bootstrap CI, PR-AUC, balanced accuracy, sensitivity, specificity, F1 and MCC. A DeLong test compares the AUCs. Accuracy alone is misleading with imbalanced classes.
6. Biological analyses: SHAP, early-vs-late differential expression (Mann–Whitney, BH-FDR), a co-expression network on training samples, and KEGG enrichment.

In [ ]:
!pip -q install imbalanced-learn xgboost shap gseapy reportlab python-docx networkx

In [ ]:
# ===================== 1. SETTINGS (check before running) =====================
DATA_FILE = "denseDataOnlyDownload-3.tsv"
DEMO = False          # leave False. True only stamps "DEMO" on every page for test runs.

TITLE = ("An Interpretable Machine Learning Framework for Early- versus Late-Stage "
         "Classification of Breast Cancer from Gene Expression, with Co-expression "
         "Network and Pathway Analysis")
AUTHORS = ("Sriporna Biswas<sup>1*</sup>, Tanjidul Huda<sup>2</sup>, Hema Pal<sup>3</sup> "
           "and Vinod Kumar Gupta<sup>3</sup>")
AFFILIATIONS = [
    "<sup>1</sup>Department of Computer Science &amp; Engineering, Chandigarh University, Punjab, India 140413",
    "<sup>2</sup>Department of Biomedical Science, BCAS, University of Delhi, Delhi, India 110007",
    "<sup>3</sup>Rapture Biotech International (P) Ltd., D-201, Sector 10, Noida (U.P.), India 201301",
]
CORRESPONDING = "*Corresponding author. E-mail: sripornab13@gmail.com"
FUNDING = ("This research received no specific grant from any funding agency in the public, "
           "commercial or not-for-profit sectors.")          # edit if you had funding
CODE_AVAILABILITY = ("The complete analysis notebook (Python) is available from the "
                     "corresponding author on reasonable request.")  # or put your GitHub link
TEMPLATE_DOCX = "Sriporna_Paper_for_Publication_SSS-3.docx"  # optional: your formatted Word file; its styles and page setup are reused
SEED = 42
OUT = "results"

In [ ]:
# ===================== 2. Imports =====================
import os, json, re, math, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from scipy import stats
from scipy.cluster.hierarchy import linkage, leaves_list
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (train_test_split, StratifiedKFold, GridSearchCV,
                                     cross_val_score, cross_val_predict)
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve, confusion_matrix,
                             balanced_accuracy_score, recall_score, precision_score,
                             f1_score, matthews_corrcoef, accuracy_score)
try:                                   # imbalanced-learn (Colab); built-in SMOTE otherwise
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline
    HAVE_IMBLEARN = True
except ImportError:
    HAVE_IMBLEARN = False
    from sklearn.base import BaseEstimator
    from sklearn.neighbors import NearestNeighbors
    from sklearn.pipeline import Pipeline as _SkPipeline
    from sklearn.utils._available_if import available_if
    class SMOTE(BaseEstimator):
        """SMOTE (Chawla et al., 2002): interpolate between minority samples and their k nearest minority neighbours."""
        def __init__(self, random_state=None, k_neighbors=5):
            self.random_state = random_state; self.k_neighbors = k_neighbors
        def fit_resample(self, X, y):
            X = np.asarray(X, dtype=float); y = np.asarray(y); rng = np.random.default_rng(self.random_state)
            cls, cnt = np.unique(y, return_counts=True); mino = cls[np.argmin(cnt)]; need = cnt.max() - cnt.min()
            Xm = X[y == mino]
            nn = NearestNeighbors(n_neighbors=min(self.k_neighbors + 1, len(Xm))).fit(Xm)
            idx = nn.kneighbors(Xm, return_distance=False)
            a = rng.integers(0, len(Xm), need); b_ = idx[a, rng.integers(1, idx.shape[1], need)]
            new = Xm[a] + rng.random((need, 1)) * (Xm[b_] - Xm[a])
            return np.vstack([X, new]), np.concatenate([y, np.full(need, mino)])
    class Pipeline(_SkPipeline):
        """Pipeline whose resampling steps act only during fit (same semantics as imblearn.pipeline.Pipeline)."""
        def fit(self, X, y=None, **kw):
            Xt, yt = X, y
            for _, st in self.steps[:-1]:
                if hasattr(st, "fit_resample"): Xt, yt = st.fit_resample(Xt, yt)
                else: Xt = st.fit_transform(Xt, yt)
            self.steps[-1][1].fit(Xt, yt); return self
        def _tx(self, X):
            for _, st in self.steps[:-1]:
                if not hasattr(st, "fit_resample"): X = st.transform(X)
            return X
        def predict_proba(self, X, **kw): return self.steps[-1][1].predict_proba(self._tx(X))
        def predict(self, X, **kw): return self.steps[-1][1].predict(self._tx(X))
        @available_if(lambda self: hasattr(self.steps[-1][1], "decision_function"))
        def decision_function(self, X, **kw): return self.steps[-1][1].decision_function(self._tx(X))
        @property
        def classes_(self): return self.steps[-1][1].classes_
try:                                   # XGBoost (Colab); scikit-learn gradient boosting otherwise
    from xgboost import XGBClassifier
    HAVE_XGB = True
except ImportError:
    HAVE_XGB = False
    from sklearn.ensemble import HistGradientBoostingClassifier
BOOST = "XGBoost" if HAVE_XGB else "Gradient Boosting"
import networkx as nx
warnings.filterwarnings("ignore")
os.makedirs(OUT, exist_ok=True)
np.random.seed(SEED)

# Figure style for print: one consistent look, thin marks, recessive grid
EARLY_C, LATE_C = "#2a78d6", "#eb6834"
MODEL_C = {"Logistic Regression": "#2a78d6", "SVM (RBF)": "#eb6834",
           "Random Forest": "#1baf7a", BOOST: "#6250d6"}
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9, "axes.titlesize": 10,
    "axes.labelsize": 9, "axes.edgecolor": INK2, "axes.labelcolor": INK,
    "xtick.color": INK2, "ytick.color": INK2, "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.6, "legend.frameon": False, "savefig.dpi": 300,
    "savefig.bbox": "tight", "lines.linewidth": 2,
})
R = {}   # every number the paper uses is stored here
import sys, sklearn
R["boost"] = BOOST
R["versions"] = {"python": sys.version.split()[0], "scikit-learn": sklearn.__version__,
                 "imbalanced-learn": (__import__("imblearn").__version__ if HAVE_IMBLEARN else None),
                 "xgboost": (__import__("xgboost").__version__ if HAVE_XGB else None),
                 "networkx": nx.__version__, "pandas": pd.__version__}

In [ ]:
if not os.path.exists(DATA_FILE):
    from google.colab import files
    print("Please upload denseDataOnlyDownload-3.tsv")
    _up = files.upload()
    DATA_FILE = list(_up.keys())[0]

In [ ]:
# ===================== 3. Load data and define the cohort =====================
EARLY = {"Stage I", "Stage IA", "Stage IB", "Stage II", "Stage IIA", "Stage IIB"}
LATE = {"Stage III", "Stage IIIA", "Stage IIIB", "Stage IIIC", "Stage IV"}

raw = pd.read_csv(DATA_FILE, sep="\t")
id_col = "sample" if "sample" in raw.columns else raw.columns[0]
stage_col = "pathologic_stage"
meta_cols = [c for c in [id_col, "samples", stage_col] if c in raw.columns]
GENES_ALL = [c for c in raw.columns if c not in meta_cols]

raw[id_col] = raw[id_col].astype(str)
raw["sample_type"] = raw[id_col].str[13:15]
R["n_raw"] = int(len(raw))
R["sample_types"] = {str(k): int(v) for k, v in raw["sample_type"].value_counts().items()}

d = raw[raw["sample_type"] == "01"].copy()
R["n_primary"] = int(len(d))
d["patient"] = d[id_col].str[:12]
d = d.drop_duplicates("patient")
R["n_primary_unique"] = int(len(d))

d["stage"] = d[stage_col].fillna("Missing").astype(str).str.strip()
d["y"] = d["stage"].map(lambda s: 0 if s in EARLY else (1 if s in LATE else np.nan))
R["excluded_stage"] = {k: int(v) for k, v in d.loc[d["y"].isna(), "stage"].value_counts().items()}
d = d.dropna(subset=["y"]).reset_index(drop=True)
y = d["y"].astype(int).values

X = d[GENES_ALL].apply(pd.to_numeric, errors="coerce")
miss = X.isna().mean()
GENES = [g for g in GENES_ALL if miss[g] <= 0.20]
R["genes_dropped_missing"] = [g for g in GENES_ALL if g not in GENES]
X = X[GENES]
R["missing_cells_pct"] = round(100 * float(X.isna().values.mean()), 2)

R["n"] = int(len(y)); R["n_early"] = int((y == 0).sum()); R["n_late"] = int((y == 1).sum())
R["pct_late"] = round(100 * y.mean(), 1); R["k"] = len(GENES)
R["stage_detail"] = {k: int(v) for k, v in d["stage"].value_counts().items()}
print(json.dumps({k: R[k] for k in ["n_raw", "sample_types", "n_primary", "n_primary_unique",
                                     "excluded_stage", "n", "n_early", "n_late", "pct_late", "k"]}, indent=1))

# Figure 2: stage distribution
order = ["Stage I", "Stage IA", "Stage IB", "Stage II", "Stage IIA", "Stage IIB",
         "Stage III", "Stage IIIA", "Stage IIIB", "Stage IIIC", "Stage IV"]
cnt = [(s, R["stage_detail"].get(s, 0)) for s in order if R["stage_detail"].get(s, 0) > 0]
fig, ax = plt.subplots(figsize=(6.4, 2.8))
bars = ax.bar([s.replace("Stage ", "") for s, _ in cnt], [c for _, c in cnt], width=0.7,
              color=[EARLY_C if s in EARLY else LATE_C for s, _ in cnt], edgecolor="white", linewidth=1)
for b, (_, c) in zip(bars, cnt):
    ax.text(b.get_x() + b.get_width() / 2, c, str(c), ha="center", va="bottom", fontsize=7.5, color=INK2)
ax.set_xlabel("AJCC pathologic stage"); ax.set_ylabel("Primary tumours")
ax.grid(axis="x", visible=False)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=EARLY_C, label=f"Early, I–II (n = {R['n_early']})"),
                   Patch(color=LATE_C, label=f"Late, III–IV (n = {R['n_late']})")], loc="upper right")
fig.savefig(f"{OUT}/fig2_stage_distribution.png"); plt.close(fig)

In [ ]:
# ===================== 4. Train/test split and model training =====================
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)
R["n_train"], R["n_test"] = int(len(y_tr)), int(len(y_te))
R["n_train_early"], R["n_train_late"] = int((y_tr == 0).sum()), int((y_tr == 1).sum())
R["n_test_early"], R["n_test_late"] = int((y_te == 0).sum()), int((y_te == 1).sum())
R["acc_all_early_test"] = round(100 * (y_te == 0).mean(), 1)
ratio = (y_tr == 0).sum() / (y_tr == 1).sum()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

MODELS = {
    "Logistic Regression": (LogisticRegression(max_iter=5000, random_state=SEED),
                            {"clf__C": [0.001, 0.01, 0.1, 1, 10]}),
    "SVM (RBF)": (SVC(kernel="rbf", probability=True, random_state=SEED),
                  {"clf__C": [0.1, 1, 10], "clf__gamma": ["scale", 0.01, 0.001]}),
    "Random Forest": (RandomForestClassifier(n_estimators=500, random_state=SEED, n_jobs=-1),
                      {"clf__max_depth": [None, 5, 10], "clf__min_samples_leaf": [1, 5]}),
    BOOST: ((XGBClassifier(eval_metric="logloss", random_state=SEED, n_jobs=-1,
                           subsample=0.8, colsample_bytree=0.8),
             {"clf__n_estimators": [200, 400], "clf__max_depth": [2, 3, 5],
              "clf__learning_rate": [0.03, 0.1]}) if HAVE_XGB else
            (HistGradientBoostingClassifier(random_state=SEED, early_stopping=False),
             {"clf__max_iter": [200, 400], "clf__max_depth": [2, 3, 5],
              "clf__learning_rate": [0.03, 0.1]})),
}

def youden(yt, p):
    fpr, tpr, thr = roc_curve(yt, p)
    return float(thr[np.argmax(tpr - fpr)])

def metrics_at(yt, p, t):
    pr = (p >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(yt, pr, labels=[0, 1]).ravel()
    return {"ROC-AUC": roc_auc_score(yt, p), "PR-AUC": average_precision_score(yt, p),
            "Balanced accuracy": balanced_accuracy_score(yt, pr),
            "Sensitivity": recall_score(yt, pr, zero_division=0),
            "Specificity": tn / (tn + fp) if tn + fp else 0.0,
            "Precision": precision_score(yt, pr, zero_division=0),
            "F1": f1_score(yt, pr, zero_division=0), "MCC": matthews_corrcoef(yt, pr),
            "Accuracy": accuracy_score(yt, pr), "Threshold": t,
            "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)}

def boot_ci(yt, p, t, n=1000):
    rng = np.random.default_rng(SEED); a, b = [], []
    yt, p = np.asarray(yt), np.asarray(p)
    for _ in range(n):
        i = rng.integers(0, len(yt), len(yt))
        if len(np.unique(yt[i])) < 2: continue
        a.append(roc_auc_score(yt[i], p[i])); b.append(balanced_accuracy_score(yt[i], (p[i] >= t).astype(int)))
    return np.percentile(a, [2.5, 97.5]), np.percentile(b, [2.5, 97.5])

# DeLong test for two correlated ROC curves (DeLong et al., 1988; fast algorithm of Sun & Xu, 2014)
def _midrank(x):
    j = np.argsort(x); z = x[j]; n = len(x); t = np.zeros(n); i = 0
    while i < n:
        k = i
        while k < n and z[k] == z[i]: k += 1
        t[i:k] = 0.5 * (i + k - 1) + 1; i = k
    out = np.empty(n); out[j] = t; return out

def delong_test(yt, p1, p2):
    yt = np.asarray(yt); o = np.argsort(-yt, kind="mergesort")
    P = np.vstack([np.asarray(p1)[o], np.asarray(p2)[o]]); m = int(yt.sum()); n = len(yt) - m
    tx = np.array([_midrank(r[:m]) for r in P]); ty = np.array([_midrank(r[m:]) for r in P])
    tz = np.array([_midrank(r) for r in P])
    aucs = tz[:, :m].sum(1) / m / n - (m + 1.0) / (2.0 * n)
    v01 = (tz[:, :m] - tx) / n; v10 = 1.0 - (tz[:, m:] - ty) / m
    S = np.cov(v01) / m + np.cov(v10) / n
    var = S[0, 0] + S[1, 1] - 2 * S[0, 1]
    z = (aucs[0] - aucs[1]) / math.sqrt(var) if var > 0 else 0.0
    return float(aucs[0]), float(aucs[1]), float(2 * stats.norm.sf(abs(z)))

def make_pipe(est, smote=True):
    steps = [("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]
    if smote: steps.append(("smote", SMOTE(random_state=SEED, k_neighbors=5)))
    steps.append(("clf", est))
    return Pipeline(steps)

FIT, PROB, ROWS, PARAMS, ROWS_CW = {}, {}, [], {}, []
for name, (est, grid) in MODELS.items():
    gs = GridSearchCV(make_pipe(est), grid, scoring="roc_auc", cv=cv, n_jobs=-1).fit(X_tr, y_tr)
    best = gs.best_estimator_
    cv_auc = cross_val_score(best, X_tr, y_tr, cv=cv, scoring="roc_auc")
    oof = cross_val_predict(best, X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]
    t = youden(y_tr, oof)
    p = best.predict_proba(X_te)[:, 1]
    m = metrics_at(y_te, p, t)
    (alo, ahi), (blo, bhi) = boot_ci(y_te, p, t)
    m.update({"AUC_lo": alo, "AUC_hi": ahi, "BA_lo": blo, "BA_hi": bhi,
              "CV_AUC_mean": cv_auc.mean(), "CV_AUC_sd": cv_auc.std(), "Model": name})
    ROWS.append(m); FIT[name] = best; PROB[name] = p
    PARAMS[name] = {"grid": {k.replace("clf__", ""): v for k, v in grid.items()},
                    "best": {k.replace("clf__", ""): v for k, v in gs.best_params_.items()}}
    # sensitivity analysis: class weighting instead of SMOTE
    est_cw = MODELS[name][0].__class__(**MODELS[name][0].get_params())
    if name == BOOST and HAVE_XGB: est_cw.set_params(scale_pos_weight=ratio)
    else: est_cw.set_params(class_weight="balanced")
    gs_cw = GridSearchCV(make_pipe(est_cw, smote=False), grid, scoring="roc_auc", cv=cv, n_jobs=-1).fit(X_tr, y_tr)
    oof_cw = cross_val_predict(gs_cw.best_estimator_, X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]
    p_cw = gs_cw.best_estimator_.predict_proba(X_te)[:, 1]
    mc = metrics_at(y_te, p_cw, youden(y_tr, oof_cw)); mc["Model"] = name; ROWS_CW.append(mc)
    print(f"{name:20s} test AUC {m['ROC-AUC']:.3f} [{alo:.3f}-{ahi:.3f}]  BA {m['Balanced accuracy']:.3f}  "
          f"Sens {m['Sensitivity']:.3f} Spec {m['Specificity']:.3f} | CV AUC {cv_auc.mean():.3f}±{cv_auc.std():.3f}")

RES = pd.DataFrame(ROWS).set_index("Model")
RES_CW = pd.DataFrame(ROWS_CW).set_index("Model")
BEST = RES["ROC-AUC"].idxmax()
R["best"] = BEST
R["delong"] = {}
for name in RES.index:
    if name != BEST:
        _, _, pv = delong_test(y_te, PROB[BEST], PROB[name]); R["delong"][name] = pv
RES.round(4).to_csv(f"{OUT}/table2_test_performance.csv")
RES_CW.round(4).to_csv(f"{OUT}/tableS3_class_weight_no_smote.csv")
print("\nBest model by test ROC-AUC:", BEST, "| DeLong p vs others:", {k: round(v, 3) for k, v in R["delong"].items()})

In [ ]:
# ===================== 5. ROC / PR curves and confusion matrix =====================
fig, axs = plt.subplots(1, 2, figsize=(7.2, 3.4))
for name in RES.sort_values("ROC-AUC", ascending=False).index:
    fpr, tpr, _ = roc_curve(y_te, PROB[name]); pr, rc, _ = precision_recall_curve(y_te, PROB[name])
    axs[0].plot(fpr, tpr, color=MODEL_C[name], label=f"{name} ({RES.loc[name, 'ROC-AUC']:.3f})")
    axs[1].plot(rc, pr, color=MODEL_C[name], label=f"{name} ({RES.loc[name, 'PR-AUC']:.3f})")
axs[0].plot([0, 1], [0, 1], ls="--", lw=1, color=INK2)
axs[1].axhline(y_te.mean(), ls="--", lw=1, color=INK2)
axs[0].set(xlabel="False positive rate (1 − specificity)", ylabel="True positive rate (sensitivity)",
           title="(a) ROC curves, AUC in brackets", xlim=(0, 1), ylim=(0, 1.01))
axs[1].set(xlabel="Recall (sensitivity)", ylabel="Precision", title="(b) Precision–recall, AP in brackets",
           xlim=(0, 1), ylim=(0, 1.01))
for a in axs: a.legend(fontsize=7, loc="lower right" if a is axs[0] else "upper right"); a.set_aspect("equal")
fig.tight_layout(); fig.savefig(f"{OUT}/fig3_roc_pr.png"); plt.close(fig)

mb = RES.loc[BEST]
cm = np.array([[mb["TN"], mb["FP"]], [mb["FN"], mb["TP"]]]).astype(int)
fig, ax = plt.subplots(figsize=(3.2, 3.0))
ax.imshow(cm, cmap="Blues"); ax.grid(False)
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i, j]}\n({100 * cm[i, j] / cm[i].sum():.0f}%)", ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else INK, fontsize=10)
ax.set_xticks([0, 1], ["Early", "Late"]); ax.set_yticks([0, 1], ["Early", "Late"])
ax.set_xlabel("Predicted stage"); ax.set_ylabel("True stage")
ax.set_title(f"{BEST}, threshold {mb['Threshold']:.2f}")
fig.savefig(f"{OUT}/fig4_confusion_matrix.png"); plt.close(fig)

In [ ]:
# ===================== 6. Model interpretation: SHAP (permutation importance if SHAP is unavailable) =====================
def transform_only(pipe, Xdf):
    Z = Xdf
    for nm, st in pipe.steps[:-1]:
        if hasattr(st, "fit_resample"): continue
        Z = st.transform(Z)
    return pd.DataFrame(Z, columns=Xdf.columns, index=Xdf.index)

xgb_pipe = FIT[BOOST]
Z_te = transform_only(xgb_pipe, X_te)
R["shap_method"] = "SHAP"
try:
    import shap
    sv = shap.TreeExplainer(xgb_pipe.named_steps["clf"]).shap_values(Z_te)
    if isinstance(sv, list): sv = sv[1]
    sv = np.asarray(sv)
    if sv.ndim == 3: sv = sv[..., 1]
    imp = pd.Series(np.abs(sv).mean(0), index=GENES)
    direction = pd.Series([np.corrcoef(Z_te[g], sv[:, i])[0, 1] if Z_te[g].std() > 0 else 0
                           for i, g in enumerate(GENES)], index=GENES)
    plt.figure()
    shap.summary_plot(sv, Z_te, show=False, max_display=15, plot_size=(6.4, 5.2))
    plt.gcf().savefig(f"{OUT}/fig5_shap.png"); plt.close("all")
except Exception as e:
    print("SHAP not available, using permutation importance:", e)
    from sklearn.inspection import permutation_importance
    R["shap_method"] = "permutation"
    pi = permutation_importance(xgb_pipe, X_te, y_te, scoring="roc_auc", n_repeats=30, random_state=SEED)
    imp = pd.Series(np.clip(pi.importances_mean, 0, None), index=GENES)
    p_hi = xgb_pipe.predict_proba(X_te)[:, 1]
    direction = pd.Series([stats.spearmanr(X_te[g], p_hi, nan_policy="omit")[0] for g in GENES], index=GENES)
    top = imp.sort_values().tail(15)
    fig, ax = plt.subplots(figsize=(6.4, 4.6))
    ax.barh(top.index, top.values, color=[LATE_C if direction[g] > 0 else EARLY_C for g in top.index], height=0.6)
    ax.set_xlabel("Mean decrease in test ROC-AUC when permuted"); ax.grid(axis="y", visible=False)
    fig.savefig(f"{OUT}/fig5_shap.png"); plt.close(fig)
imp = imp.sort_values(ascending=False)
R["top_shap"] = list(imp.index[:10])
R["shap_dir"] = {g: ("late" if direction[g] > 0 else "early") for g in imp.index[:10]}
pd.DataFrame({"importance": imp, "higher_expression_pushes_towards": [("late" if direction[g] > 0 else "early") for g in imp.index]}
             ).to_csv(f"{OUT}/shap_importance.csv")
print("Top genes:", R["top_shap"])

In [ ]:
# ===================== 7. Differential expression, early vs late =====================
de = []
Xe, Xl = X[y == 0], X[y == 1]
for g in GENES:
    a, b = Xe[g].dropna(), Xl[g].dropna()
    u, p = stats.mannwhitneyu(b, a, alternative="two-sided")
    de.append({"Gene": g, "Mean early": a.mean(), "Mean late": b.mean(),
               "Difference (late − early)": b.mean() - a.mean(),
               "Effect size r": 2 * u / (len(a) * len(b)) - 1,   # rank-biserial; > 0 means higher in late stage
               "p": p})
DE = pd.DataFrame(de)
pv = DE["p"].values; o = np.argsort(pv); q = np.empty_like(pv)
q[o] = np.minimum.accumulate((pv[o] * len(pv) / np.arange(1, len(pv) + 1))[::-1])[::-1]
DE["FDR"] = np.minimum(q, 1); DE = DE.sort_values("p").reset_index(drop=True)
DE.to_csv(f"{OUT}/tableS4_differential_expression.csv", index=False)
SIG = DE[DE["FDR"] < 0.05]
R["n_sig"] = int(len(SIG)); R["sig_up"] = list(SIG[SIG["Difference (late − early)"] > 0]["Gene"])
R["sig_down"] = list(SIG[SIG["Difference (late − early)"] < 0]["Gene"])
print("Genes with FDR < 0.05:", R["n_sig"])

# Figure 6: heatmap of top genes, samples sorted by stage
hm_genes = list(DE["Gene"].head(20))
Zall = (X[hm_genes] - X[hm_genes].mean()) / X[hm_genes].std()
Zall = Zall.fillna(0).clip(-3, 3)
ordr = np.argsort(y, kind="stable")
fig = plt.figure(figsize=(7.0, 4.4))
ax0 = fig.add_axes([0.14, 0.86, 0.72, 0.035]); ax1 = fig.add_axes([0.14, 0.08, 0.72, 0.77])
cax = fig.add_axes([0.88, 0.08, 0.02, 0.4])
ax0.imshow(y[ordr][None, :], aspect="auto", cmap=matplotlib.colors.ListedColormap([EARLY_C, LATE_C]))
ax0.set_xticks([]); ax0.set_yticks([0], ["Stage"]); ax0.grid(False)
for s in ax0.spines.values(): s.set_visible(False)
ax0.text(R["n_early"] / 2, -0.9, "Early (I–II)", ha="center", fontsize=8, color=INK)
ax0.text(R["n_early"] + R["n_late"] / 2, -0.9, "Late (III–IV)", ha="center", fontsize=8, color=INK)
im = ax1.imshow(Zall.values[ordr].T, aspect="auto", cmap="RdBu_r", vmin=-3, vmax=3, interpolation="nearest")
ax1.set_yticks(range(len(hm_genes)), hm_genes, fontsize=7); ax1.set_xticks([]); ax1.grid(False)
ax1.set_xlabel(f"Primary tumours (n = {R['n']}), sorted by stage group")
ax1.axvline(R["n_early"] - 0.5, color=INK, lw=1)
fig.colorbar(im, cax=cax).set_label("Expression z-score", fontsize=8)
fig.savefig(f"{OUT}/fig6_heatmap.png"); plt.close(fig)

In [ ]:
# ===================== 8. Co-expression network (training samples only) =====================
C = X_tr.corr(method="pearson")
Lk = linkage(C.fillna(0).values, "average"); lo = leaves_list(Lk)
Co = C.iloc[lo, lo]
fig, ax = plt.subplots(figsize=(7.0, 6.2))
im = ax.imshow(Co.values, cmap="RdBu_r", vmin=-1, vmax=1); ax.grid(False)
ax.set_xticks(range(len(Co)), Co.columns, rotation=90, fontsize=5.5)
ax.set_yticks(range(len(Co)), Co.index, fontsize=5.5)
fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02).set_label("Pearson r", fontsize=8)
fig.savefig(f"{OUT}/fig7_correlation.png"); plt.close(fig)

NET = {}
for thr in (0.7, 0.5, 0.3):
    G = nx.Graph(); G.add_nodes_from(GENES)
    for i, a in enumerate(GENES):
        for b in GENES[i + 1:]:
            r = C.loc[a, b]
            if pd.notna(r) and abs(r) >= thr: G.add_edge(a, b, r=float(r))
    NET[thr] = G
R["net"] = {str(t): {"edges": NET[t].number_of_edges(),
                     "nodes": sum(1 for _, k in NET[t].degree() if k > 0)} for t in NET}
R["edges07"] = [(a, b, round(dd["r"], 2)) for a, b, dd in NET[0.7].edges(data=True)]
HUB_T = 0.5 if NET[0.5].number_of_edges() >= 10 else 0.3
R["hub_thr"] = HUB_T
Gh = NET[HUB_T]
deg = pd.Series(dict(Gh.degree())).sort_values(ascending=False)
deg = deg[deg > 0]
R["hubs"] = [(g, int(k)) for g, k in deg.head(10).items()]
pd.DataFrame({"degree": deg}).to_csv(f"{OUT}/hub_genes.csv")

fig, axs = plt.subplots(1, 2, figsize=(7.6, 4.0), gridspec_kw={"width_ratios": [1.5, 1]})
Gs = Gh.subgraph([g for g in Gh if Gh.degree(g) > 0]).copy()
if Gs.number_of_nodes():
    order_nodes = [g for g in Co.columns if g in Gs]          # clustering order keeps modules together
    ang = {g: 2 * math.pi * i / len(order_nodes) for i, g in enumerate(order_nodes)}
    pos = {g: (math.cos(a), math.sin(a)) for g, a in ang.items()}
    ecol = [LATE_C if dd["r"] > 0 else EARLY_C for _, _, dd in Gs.edges(data=True)]
    nx.draw_networkx_edges(Gs, pos, ax=axs[0], edge_color=ecol, width=[0.6 + 2 * (abs(dd["r"]) - HUB_T) for _, _, dd in Gs.edges(data=True)], alpha=0.7)
    hubset = set(g for g, _ in R["hubs"][:5])
    nx.draw_networkx_nodes(Gs, pos, ax=axs[0], node_size=[20 + 12 * Gs.degree(g) for g in Gs],
                           node_color=["#fcd9c6" if g in hubset else "#dfe9f7" for g in Gs], edgecolors=INK2, linewidths=0.6)
    for g, a in ang.items():
        deg_a = math.degrees(a); flip = 90 < deg_a < 270
        axs[0].text(1.13 * math.cos(a), 1.13 * math.sin(a), g, fontsize=6.3, color=INK,
                    rotation=deg_a + 180 if flip else deg_a, rotation_mode="anchor",
                    ha="right" if flip else "left", va="center",
                    fontweight="bold" if g in hubset else "normal")
    axs[0].set_xlim(-1.55, 1.55); axs[0].set_ylim(-1.55, 1.55); axs[0].set_aspect("equal")
axs[0].set_title(f"(a) Co-expression network, |r| ≥ {HUB_T}"); axs[0].axis("off")
top10 = deg.head(10)[::-1]
axs[1].barh(top10.index, top10.values, color=LATE_C, height=0.6)
for i, v in enumerate(top10.values): axs[1].text(v, i, f" {v}", va="center", fontsize=7.5, color=INK2)
axs[1].set_xlabel("Degree (number of co-expressed partners)"); axs[1].set_title("(b) Top hub genes")
axs[1].grid(axis="y", visible=False)
fig.tight_layout(); fig.savefig(f"{OUT}/fig8_network_hubs.png"); plt.close(fig)
print("Network:", R["net"], "| hub threshold:", HUB_T, "| hubs:", R["hubs"][:5])

In [ ]:
# ===================== 9. KEGG enrichment (needs internet; works in Colab) =====================
R["kegg_ok"] = False
KEGG_GENES = list(dict.fromkeys(list(imp.index[:20]) + list(SIG["Gene"])))[:30]
R["kegg_input"] = KEGG_GENES
try:
    import gseapy as gp
    enr = gp.enrichr(gene_list=KEGG_GENES, gene_sets=["KEGG_2021_Human"], organism="human", outdir=None)
    K = enr.results[["Term", "Overlap", "P-value", "Adjusted P-value", "Combined Score", "Genes"]]
    K = K.sort_values("Adjusted P-value").reset_index(drop=True)
    K.to_csv(f"{OUT}/tableS5_kegg.csv", index=False)
    R["kegg_ok"] = True
    R["kegg_n_sig"] = int((K["Adjusted P-value"] < 0.05).sum())
    R["kegg_top"] = [(r.Term, r.Overlap, float(r["Adjusted P-value"]), r.Genes) for _, r in K.head(10).iterrows()]
    k10 = K.head(12)[::-1]
    fig, ax = plt.subplots(figsize=(6.6, 3.8))
    ax.barh(k10["Term"], -np.log10(k10["Adjusted P-value"]), color=EARLY_C, height=0.6)
    ax.axvline(-np.log10(0.05), ls="--", lw=1, color=INK2)
    for i, (v, ov) in enumerate(zip(-np.log10(k10["Adjusted P-value"]), k10["Overlap"])):
        ax.text(v, i, f" {ov}", va="center", fontsize=7, color=INK2)
    ax.set_xlabel("−log10 (BH-adjusted p)   [dashed line: FDR = 0.05; labels: overlap]")
    ax.tick_params(axis="y", labelsize=7.5); ax.grid(axis="y", visible=False)
    fig.savefig(f"{OUT}/fig9_kegg.png"); plt.close(fig)
    print("KEGG terms with FDR < 0.05:", R["kegg_n_sig"])
except Exception as e:
    print("KEGG enrichment skipped:", e)

In [ ]:
# ===================== 10. Workflow figure (Figure 1) =====================
def workflow_fig(path):
    fig, ax = plt.subplots(figsize=(7.2, 6.0)); ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.axis("off")
    def box(x, y0, w, h, txt, hi=False):
        ax.add_patch(FancyBboxPatch((x, y0), w, h, boxstyle="round,pad=0.3,rounding_size=1.5",
                     fc="#dfe9f7" if hi else "white", ec=EARLY_C if hi else INK2, lw=1.4 if hi else 0.9))
        ax.text(x + w / 2, y0 + h / 2, txt, ha="center", va="center", fontsize=7, color=INK, linespacing=1.35)
    def arr(x1, y1, x2, y2):
        ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="-|>", color=INK2, lw=0.9))
    box(12, 91, 76, 7, f"TCGA-BRCA expression + clinical data (UCSC Xena)\n{R['k']} cancer-related genes")
    box(12, 79, 76, 7, f"Primary tumours, one per patient\nearly I–II (n = {R['n_early']})  vs  late III–IV (n = {R['n_late']})")
    box(12, 67, 76, 7, f"Stratified 80/20 split\ntraining n = {R['n_train']}  |  held-out test n = {R['n_test']}", hi=True)
    kg = "\n→ KEGG enrichment" if R["kegg_ok"] else ""
    interp = "SHAP" if R["shap_method"] == "SHAP" else "Permutation-importance"
    box(1, 34, 47, 26, f"Training set, stratified 5-fold CV\nmedian imputation → z-scaling\n→ SMOTE (training folds only)\nLR | SVM | RF | {BOOST}\ngrid search on ROC-AUC\nYouden threshold from\nout-of-fold predictions")
    box(52, 34, 47, 26, "Biological analyses\nearly vs late differential expression\n(Mann–Whitney, BH-FDR)\nco-expression network\n(training samples) → hub genes" + kg)
    box(1, 15, 47, 12, "Single evaluation on the test set\nROC-AUC (95% CI), PR-AUC, balanced\naccuracy, sensitivity, specificity, MCC;\nDeLong test")
    box(1, 1, 47, 8, f"{interp} explanation of the\n{BOOST} model on the test set")
    box(52, 12, 47, 15, "Integration\ngenes ranked by " + ("SHAP" if interp == "SHAP" else "importance") + ", differential\nexpression and network degree" + ("\n→ pathway context" if R["kegg_ok"] else ""))
    arr(50, 91, 50, 86.4); arr(50, 79, 50, 74.4); arr(40, 67, 24.5, 60.4); arr(60, 67, 75.5, 60.4)
    arr(24.5, 34, 24.5, 27.4); arr(24.5, 15, 24.5, 9.4); arr(75.5, 34, 75.5, 27.4); arr(48.3, 5, 55, 11.6)
    fig.savefig(path); plt.close(fig)
workflow_fig(f"{OUT}/fig1_workflow.png")

with open(f"{OUT}/results.json", "w") as fh:
    json.dump(R, fh, indent=1, default=str)
print("All analyses done.")

In [ ]:
# ===================== 11. Write the paper (all numbers come from R) =====================
import html as _html

# ---------- references (verified) ----------
REFS = {
 "bray": "Bray F, Laversanne M, Sung H, Ferlay J, Siegel RL, Soerjomataram I, Jemal A. Global cancer statistics 2022: GLOBOCAN estimates of incidence and mortality worldwide for 36 cancers in 185 countries. CA Cancer J Clin. 2024;74(3):229–263.",
 "ajcc": "Giuliano AE, Edge SB, Hortobagyi GN. Eighth edition of the AJCC Cancer Staging Manual: breast cancer. Ann Surg Oncol. 2018;25(7):1783–1785.",
 "tcga": "Cancer Genome Atlas Network. Comprehensive molecular portraits of human breast tumours. Nature. 2012;490(7418):61–70.",
 "parker": "Parker JS, Mullins M, Cheang MCU, Leung S, Voduc D, Vickery T, et al. Supervised risk predictor of breast cancer based on intrinsic subtypes. J Clin Oncol. 2009;27(8):1160–1167.",
 "kourou": "Kourou K, Exarchos TP, Exarchos KP, Karamouzis MV, Fotiadis DI. Machine learning applications in cancer prognosis and prediction. Comput Struct Biotechnol J. 2015;13:8–17.",
 "libbrecht": "Libbrecht MW, Noble WS. Machine learning applications in genetics and genomics. Nat Rev Genet. 2015;16(6):321–332.",
 "tabassum": "Tabassum N, Kamal MAS, Akhand MAH, Yamada K. Cancer classification from gene expression using ensemble learning with an influential feature selection technique. BioMedInformatics. 2024;4(2):1275–1288.",
 "alharbi": "Alharbi F, Vakanski A, Zhang B, Elbashir MK, Mohammed M. Comparative analysis of multi-omics integration using graph neural networks for cancer classification. IEEE Access. 2025;13:37724–37736. doi:10.1109/ACCESS.2025.3540769",
 "patel": "Patel JC, Shakyawar SK, Sethi S, Guda C. GAIN-BRCA: a graph-based AI-net framework for breast cancer subtype classification using multiomics data. Bioinform Adv. 2025;5(1):vbaf116. doi:10.1093/bioadv/vbaf116",
 "wang": "Wang Y, Liu ZP. Identifying biomarkers for breast cancer by gene regulatory network rewiring. BMC Bioinformatics. 2022. doi:10.1186/s12859-021-04225-1",
 "saito": "Saito T, Rehmsmeier M. The precision-recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets. PLoS One. 2015;10(3):e0118432.",
 "chicco": "Chicco D, Jurman G. The advantages of the Matthews correlation coefficient (MCC) over F1 score and accuracy in binary classification evaluation. BMC Genomics. 2020;21:6.",
 "chawla": "Chawla NV, Bowyer KW, Hall LO, Kegelmeyer WP. SMOTE: synthetic minority over-sampling technique. J Artif Intell Res. 2002;16:321–357.",
 "kaufman": "Kaufman S, Rosset S, Perlich C, Stitelman O. Leakage in data mining: formulation, detection, and avoidance. ACM Trans Knowl Discov Data. 2012;6(4):15.",
 "blagus": "Blagus R, Lusa L. SMOTE for high-dimensional class-imbalanced data. BMC Bioinformatics. 2013;14:106.",
 "lundberg2017": "Lundberg SM, Lee SI. A unified approach to interpreting model predictions. Adv Neural Inf Process Syst. 2017;30:4765–4774.",
 "lundberg2020": "Lundberg SM, Erion G, Chen H, DeGrave A, Prutkin JM, Nair B, et al. From local explanations to global understanding with explainable AI for trees. Nat Mach Intell. 2020;2(1):56–67.",
 "ecmarker": "Jin T, Nguyen ND, Talos F, Wang D. ECMarker: interpretable machine learning model identifies gene expression biomarkers predicting clinical outcomes and reveals molecular mechanisms of human disease in early stages. Bioinformatics. 2021;37(8):1115–1124.",
 "kegg": "Kanehisa M, Furumichi M, Sato Y, Kawashima M, Ishiguro-Watanabe M. KEGG for taxonomy-based analysis of pathways and genomes. Nucleic Acids Res. 2023;51(D1):D587–D592.",
 "gsea": "Subramanian A, Tamayo P, Mootha VK, Mukherjee S, Ebert BL, Gillette MA, et al. Gene set enrichment analysis: a knowledge-based approach for interpreting genome-wide expression profiles. Proc Natl Acad Sci U S A. 2005;102(43):15545–15550.",
 "xena": "Goldman MJ, Craft B, Hastie M, Repečka K, McDade F, Kamath A, et al. Visualizing and interpreting cancer genomics data via the Xena platform. Nat Biotechnol. 2020;38(6):675–678.",
 "pedregosa": "Pedregosa F, Varoquaux G, Gramfort A, Michel V, Thirion B, Grisel O, et al. Scikit-learn: machine learning in Python. J Mach Learn Res. 2011;12:2825–2830.",
 "imblearn": "Lemaître G, Nogueira F, Aridas CK. Imbalanced-learn: a Python toolbox to tackle the curse of imbalanced datasets in machine learning. J Mach Learn Res. 2017;18(17):1–5.",
 "cortes": "Cortes C, Vapnik V. Support-vector networks. Mach Learn. 1995;20(3):273–297.",
 "breiman": "Breiman L. Random forests. Mach Learn. 2001;45(1):5–32.",
 "xgb": "Chen T, Guestrin C. XGBoost: a scalable tree boosting system. In: Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining. 2016:785–794.",
 "friedman": "Friedman JH. Greedy function approximation: a gradient boosting machine. Ann Stat. 2001;29(5):1189–1232.",
 "youden": "Youden WJ. Index for rating diagnostic tests. Cancer. 1950;3(1):32–35.",
 "delong": "DeLong ER, DeLong DM, Clarke-Pearson DL. Comparing the areas under two or more correlated receiver operating characteristic curves: a nonparametric approach. Biometrics. 1988;44(3):837–845.",
 "bh": "Benjamini Y, Hochberg Y. Controlling the false discovery rate: a practical and powerful approach to multiple testing. J R Stat Soc Series B Stat Methodol. 1995;57(1):289–300.",
 "networkx": "Hagberg AA, Schult DA, Swart PJ. Exploring network structure, dynamics, and function using NetworkX. In: Proceedings of the 7th Python in Science Conference (SciPy 2008). 2008:11–15.",
 "wgcna": "Langfelder P, Horvath S. WGCNA: an R package for weighted correlation network analysis. BMC Bioinformatics. 2008;9:559.",
 "enrichr": "Kuleshov MV, Jones MR, Rouillard AD, Fernandez NF, Duan Q, Wang Z, et al. Enrichr: a comprehensive gene set enrichment analysis web server 2016 update. Nucleic Acids Res. 2016;44(W1):W90–W97.",
 "gseapy": "Fang Z, Liu X, Peltz G. GSEApy: a comprehensive package for performing gene set enrichment analysis in Python. Bioinformatics. 2023;39(1):btac757.",
}
_ORDER = []
def cite(*keys):
    nums = []
    for k in keys:
        if k not in _ORDER: _ORDER.append(k)
        nums.append(_ORDER.index(k) + 1)
    nums = sorted(set(nums)); parts = []; i = 0
    while i < len(nums):
        j = i
        while j + 1 < len(nums) and nums[j + 1] == nums[j] + 1: j += 1
        parts.append(f"{nums[i]}–{nums[j]}" if j - i >= 2 else ", ".join(str(n) for n in nums[i:j + 1])); i = j + 1
    return "[" + ", ".join(parts) + "]"

# ---------- short, factual gene and pathway notes used in the Discussion ----------
GENE_INFO = {
 "TP53": "the p53 tumour suppressor, a transcription factor controlling cell-cycle arrest, DNA repair and apoptosis",
 "BRCA1": "a homologous-recombination DNA-repair protein whose germline mutation confers high breast cancer risk",
 "BRCA2": "a homologous-recombination DNA-repair protein whose germline mutation confers high breast cancer risk",
 "ERBB2": "the HER2 receptor tyrosine kinase, amplified in about 15–20% of breast cancers and the target of trastuzumab",
 "PIK3CA": "the catalytic subunit of phosphoinositide 3-kinase, one of the most frequently mutated genes in breast cancer",
 "MYC": "a transcription factor that drives proliferation and growth and is frequently amplified in breast cancer",
 "EGFR": "the epidermal growth factor receptor, often over-expressed in basal-like and triple-negative tumours",
 "ESR1": "oestrogen receptor alpha, the defining marker of luminal breast cancer",
 "PGR": "the progesterone receptor, an oestrogen-regulated marker of luminal tumours",
 "GATA3": "a luminal lineage transcription factor closely associated with oestrogen-receptor-positive tumours",
 "FOXA1": "a pioneer transcription factor that enables oestrogen-receptor binding to chromatin in luminal cells",
 "PDGFRA": "the platelet-derived growth factor receptor alpha, a receptor tyrosine kinase active in stromal and mesenchymal signalling",
 "ATM": "a kinase that activates the DNA damage response to double-strand breaks",
 "BRAF": "a serine/threonine kinase of the MAPK signalling cascade",
 "KRAS": "a small GTPase upstream of MAPK and PI3K signalling",
 "JUN": "a component of the AP-1 transcription factor that responds to growth-factor and stress signals",
 "FOS": "a component of the AP-1 transcription factor induced rapidly by growth-factor signalling",
 "EP300": "a histone acetyltransferase and transcriptional co-activator",
 "CDK2": "a cyclin-dependent kinase that drives the G1/S transition of the cell cycle",
 "CDK4": "a cyclin D-dependent kinase whose inhibition is standard therapy for hormone-receptor-positive advanced breast cancer",
 "CDK6": "a cyclin D-dependent kinase whose inhibition is standard therapy for hormone-receptor-positive advanced breast cancer",
 "CCND1": "cyclin D1, the regulatory partner of CDK4/6, frequently amplified in luminal breast cancer",
 "NCOR1": "a nuclear receptor co-repressor that modulates hormone-receptor signalling",
 "NCOR2": "a nuclear receptor co-repressor (SMRT) that modulates hormone-receptor signalling",
 "POU5F1": "OCT4, a pluripotency transcription factor linked to stem-like tumour cell states",
 "SOX2": "a pluripotency transcription factor linked to stem-like tumour cell states",
 "NANOG": "a pluripotency transcription factor linked to stem-like tumour cell states",
 "SOX9": "a transcription factor associated with progenitor and stem-like properties of mammary cells",
 "STAT3": "a signal transducer and transcription factor downstream of cytokine and growth-factor receptors",
 "NOTCH1": "a Notch receptor that regulates cell-fate decisions and is implicated in mammary tumorigenesis",
 "FGFR1": "a fibroblast growth factor receptor amplified in a subset of luminal breast cancers",
 "MTOR": "a kinase integrating growth signals downstream of PI3K/AKT and the target of everolimus",
 "AKT1": "a serine/threonine kinase central to PI3K signalling",
 "PTEN": "a lipid phosphatase that antagonises PI3K signalling",
 "RUNX1": "a transcription factor recurrently mutated in luminal breast cancer",
 "RUNX2": "an osteogenic transcription factor implicated in breast cancer bone metastasis",
 "VEGFA": "vascular endothelial growth factor A, the main driver of tumour angiogenesis",
 "AR": "the androgen receptor, expressed in a large fraction of breast cancers",
 "CD44": "a hyaluronan receptor and marker of breast cancer stem-like cells",
 "ALDH1A1": "an aldehyde dehydrogenase used as a marker of breast cancer stem-like cells",
 "PROM1": "CD133, a cell-surface marker of stem-like cells",
 "CDH1": "E-cadherin, an epithelial adhesion molecule typically lost in lobular carcinoma",
 "VIM": "vimentin, a mesenchymal intermediate filament and marker of epithelial–mesenchymal transition",
 "SNAI1": "an epithelial–mesenchymal transition transcription factor",
 "TWIST1": "an epithelial–mesenchymal transition transcription factor",
 "ZEB1": "an epithelial–mesenchymal transition transcription factor",
 "CDKN2A": "p16INK4a, an inhibitor of CDK4/6",
 "RB1": "the retinoblastoma protein, which restrains G1/S progression",
 "MKI67": "Ki-67, a proliferation marker used in breast cancer grading",
 "BCL2": "an anti-apoptotic protein highly expressed in luminal tumours",
 "HIF1A": "hypoxia-inducible factor 1 alpha, a regulator of the hypoxic response",
 "MMP9": "a matrix metalloproteinase involved in extracellular matrix remodelling and invasion",
 "CTNNB1": "beta-catenin, the effector of canonical Wnt signalling",
 "TGFB1": "transforming growth factor beta 1, a cytokine with context-dependent tumour-suppressive and pro-metastatic roles",
 "CHEK2": "a checkpoint kinase in the DNA damage response and a moderate-penetrance breast cancer gene",
 "PALB2": "a partner of BRCA2 in homologous-recombination repair",
 "MAP3K1": "a MAPK-pathway kinase frequently mutated in luminal breast cancer",
 "IGF1R": "the insulin-like growth factor 1 receptor",
 "MET": "the hepatocyte growth factor receptor tyrosine kinase",
 "SMAD4": "the central mediator of TGF-beta signalling",
 "NFKB1": "a subunit of the NF-kappaB transcription factor",
 "IL6": "interleukin-6, an inflammatory cytokine that activates STAT3",
 "KIT": "a receptor tyrosine kinase for stem cell factor",
 "ERBB3": "HER3, a kinase-impaired ERBB-family receptor that signals through PI3K",
 "MDM2": "an E3 ubiquitin ligase that degrades p53",
 "CCNE1": "cyclin E1, the partner of CDK2",
 "E2F1": "a transcription factor driving S-phase gene expression",
 "WNT1": "a ligand of the canonical Wnt pathway",
 "MAPK1": "ERK2, a terminal kinase of the MAPK cascade",
 "MAPK3": "ERK1, a terminal kinase of the MAPK cascade",
 "JAK2": "a Janus kinase that activates STAT signalling downstream of cytokine receptors",
 "NFKB2": "a subunit of the non-canonical NF-kappaB pathway",
 "REL": "c-Rel, a subunit of the NF-kappaB transcription factor",
 "TGFBR2": "the TGF-beta type II receptor",
 "SMAD2": "a receptor-regulated SMAD that transduces TGF-beta signals",
 "SMAD3": "a receptor-regulated SMAD that transduces TGF-beta signals",
 "FLT3": "a receptor tyrosine kinase best known for its role in haematopoiesis and acute myeloid leukaemia",
 "SRC": "a non-receptor tyrosine kinase involved in adhesion, migration and invasion",
 "ABL1": "a non-receptor tyrosine kinase, the fusion partner of BCR in chronic myeloid leukaemia",
 "CREBBP": "a histone acetyltransferase and transcriptional co-activator paralogous to EP300",
 "ARID1A": "a subunit of the SWI/SNF chromatin-remodelling complex and a frequently mutated tumour suppressor",
 "ARID1B": "a subunit of the SWI/SNF chromatin-remodelling complex",
}
KEGG_INFO = {
 "Breast cancer": "the KEGG map of oestrogen, ERBB, PI3K–AKT, Wnt and Notch signalling in breast tumours",
 "Pathways in cancer": "the KEGG overview of signalling routes shared across cancers",
 "MicroRNAs in cancer": "genes whose expression is regulated by cancer-associated microRNAs",
 "PI3K-Akt signaling pathway": "growth-factor signalling through PI3K and AKT that promotes survival and proliferation",
 "FOXO signaling pathway": "FOXO transcription factors that control cell-cycle arrest, apoptosis and oxidative-stress responses",
 "Cell cycle": "the regulation of G1/S and G2/M transitions",
 "Thyroid hormone signaling pathway": "nuclear-receptor signalling sharing co-regulators such as NCOR1/2 and EP300",
 "Endocrine resistance": "mechanisms of resistance to endocrine therapy in hormone-receptor-positive cancer",
 "ErbB signaling pathway": "signalling by EGFR/HER-family receptors",
 "MAPK signaling pathway": "the RAS–RAF–MEK–ERK cascade",
 "Signaling pathways regulating pluripotency of stem cells": "networks maintaining stem-cell self-renewal",
 "p53 signaling pathway": "p53-dependent responses to DNA damage",
 "Notch signaling pathway": "Notch receptor signalling controlling cell fate",
 "JAK-STAT signaling pathway": "cytokine signalling through JAK kinases and STAT transcription factors",
 "Estrogen signaling pathway": "oestrogen-receptor signalling",
 "Transcriptional misregulation in cancer": "transcription factors deregulated in cancer",
 "Proteoglycans in cancer": "proteoglycan-mediated growth-factor and adhesion signalling",
}

# ---------- small helpers ----------
def it(g): return f"<i>{g}</i>"
def glist(gs):
    gs = [it(g) for g in gs]
    return gs[0] if len(gs) == 1 else (", ".join(gs[:-1]) + " and " + gs[-1] if gs else "")
def plist(xs):
    xs = list(xs); return xs[0] if len(xs) == 1 else (", ".join(xs[:-1]) + " and " + xs[-1] if xs else "")
def f3(x): return f"{x:.3f}"
def pf(p): return "p < 0.001" if p < 0.001 else f"p = {p:.3f}"
def qf(q): return "&lt; 0.001" if q < 0.001 else (f"{q:.3f}" if q < 0.1 else f"{q:.2f}")
def auc_word(a):
    return "poor" if a < 0.6 else "modest" if a < 0.7 else "moderate" if a < 0.8 else "good" if a < 0.9 else "excellent"
def num(n): return f"{n:,}"

BOOST = R["boost"]; XGB = BOOST == "XGBoost"; SH = R["shap_method"] == "SHAP"
INTERP = "SHAP" if SH else "permutation importance"
KOK = bool(R["kegg_ok"] and R.get("kegg_top"))
boost_txt = "XGBoost" if XGB else "gradient boosting"
b = RES.loc[BEST]
others = [m for m in RES.index if m != BEST]
dl = R["delong"]; dl_sig = [m for m in others if dl[m] < 0.05]
above_chance = [m for m in RES.index if RES.loc[m, "AUC_lo"] > 0.5]
auc_range_other = (RES.loc[others, "ROC-AUC"].min(), RES.loc[others, "ROC-AUC"].max())
top_shap = R["top_shap"]; late_genes = [g for g in top_shap[:6] if R["shap_dir"][g] == "late"]
early_genes = [g for g in top_shap[:6] if R["shap_dir"][g] == "early"]
hubs = R["hubs"]; hub_names = [g for g, _ in hubs]
sig = list(SIG["Gene"])
overlap_shap_de = [g for g in top_shap if g in sig]
overlap_hub = [g for g in hub_names[:10] if g in set(top_shap) | set(sig)]
excl_total = sum(R["excluded_stage"].values())
excl_txt = ", ".join(f"{k.replace('[','').replace(']','')}: {v}" for k, v in R["excluded_stage"].items())
other_types = {k: v for k, v in R["sample_types"].items() if k != "01"}
TYPE_NAME = {"11": "solid normal tissue", "06": "metastatic", "02": "recurrent tumour", "05": "additional primary"}
other_txt = plist([f"{v} {TYPE_NAME.get(k, 'type ' + k)} sample{'s' if v != 1 else ''}" for k, v in other_types.items()]) or "no other sample types"
gap = abs(b["ROC-AUC"] - b["CV_AUC_mean"])
cw_best = RES_CW["ROC-AUC"].idxmax(); cw_delta = (RES_CW["ROC-AUC"] - RES["ROC-AUC"]).abs().max()
word = auc_word(b["ROC-AUC"])
from collections import Counter
_cnt = Counter(list(dict.fromkeys(top_shap[:10])) + sig + hub_names[:10])
CONV = [g for g, c in _cnt.items() if c >= 2]
EXPLOR = RES.loc[R["boost"], "AUC_lo"] <= 0.5
imp_caveat = (f" Because the AUC confidence interval of the {R['boost']} model included 0.5, these rankings should be regarded as exploratory." if EXPLOR else "")
V = R["versions"]

DOC = []
def H1(t): DOC.append(("h1", t))
def H2(t): DOC.append(("h2", t))
def P(t): DOC.append(("p", t))
def TAB(caption, header, rows, widths=None, note=None, small=False): DOC.append(("table", caption, header, rows, widths, note, small))
FIGN = [0]
def FIG(path, caption, w=1.0):
    if os.path.exists(path): FIGN[0] += 1; DOC.append(("fig", path, f"Figure {FIGN[0]}: {caption}", w))
TABN = [0]
def tcap(t): TABN[0] += 1; return f"Table {TABN[0]}: {t}"

# ---------- Title block and abstract ----------
if not KOK: TITLE = TITLE.replace("Co-expression Network and Pathway Analysis", "Co-expression Network Analysis")
DOC += [("title", TITLE), ("authors", AUTHORS)] + [("affil", a) for a in AFFILIATIONS] + [("affil", CORRESPONDING)]

if dl_sig and len(dl_sig) == len(others):
    dl_abs = f"its AUC was significantly higher than that of every other model (DeLong {pf(max(dl[m] for m in others))} or lower)"
elif dl_sig:
    dl_abs = f"its AUC was significantly higher than that of {plist(dl_sig)} (DeLong p &lt; 0.05) but not of {plist([m for m in others if m not in dl_sig])}"
else:
    dl_abs = f"AUC differences between models were not statistically significant (DeLong p ≥ {math.floor(min(dl.values()) * 100) / 100:.2f})"
kegg_abs = (f" Enrichment analysis placed the top-ranked genes in {plist([t for t, _, q, _ in R['kegg_top'][:3]])}." if KOK else "")
de_abs = (f"of the {R['k']} genes, {R['n_sig']} differed between early and late tumours (FDR &lt; 0.05), led by {glist(sig[:3])}" if R["n_sig"]
          else f"no gene differed significantly between early and late tumours after FDR correction")
if b["ROC-AUC"] < 0.70:
    concl_abs = ("Expression of a cancer-gene panel in the primary tumour carries only limited, though measurable, information about anatomical stage. "
                 "Accuracy near the majority-class rate can hide near-chance discrimination, so imbalance-robust metrics are essential. "
                 "Expression-based stage models should be combined with clinicopathological variables and validated externally.")
elif b["ROC-AUC"] < 0.80:
    concl_abs = ("Expression of a cancer-gene panel carries moderate information about anatomical stage. The interpretable framework identifies biologically plausible "
                 "stage-associated genes, but external validation and integration with clinicopathological variables are needed before any clinical use.")
else:
    concl_abs = ("Expression of a cancer-gene panel discriminates early from late stage well, and the interpretable framework identifies biologically plausible "
                 "stage-associated genes. External validation is required before clinical use.")
ABS = [
 ("Background", "Anatomical stage is the main determinant of treatment and prognosis in breast cancer, but it is unclear how much information about stage is carried by the "
  "transcriptome of the primary tumour. We developed a leakage-free, interpretable machine-learning framework to classify early- versus late-stage breast cancer "
  "from a curated cancer-gene panel and examined the biology of the genes involved."),
 ("Methods", f"RNA-seq expression of {R['k']} cancer-related genes and AJCC pathologic stage were obtained for The Cancer Genome Atlas breast invasive carcinoma cohort "
  f"(TCGA-BRCA) from the UCSC Xena platform. After restricting the data to one primary tumour per patient, {num(R['n'])} tumours were analysed ({num(R['n_early'])} early, "
  f"Stage I–II; {R['n_late']} late, Stage III–IV). Logistic regression, a support vector machine, random forest and {boost_txt} were tuned by stratified five-fold "
  "cross-validation on 80% of the data, with imputation, scaling and SMOTE oversampling fitted only within training folds and decision thresholds chosen on "
  f"out-of-fold predictions, and were evaluated once on a held-out test set (20%). Models were explained with {INTERP}, and stage-associated genes were examined by "
  + ("differential expression, a co-expression network and KEGG pathway enrichment." if KOK else "differential expression and a co-expression network.")),
 ("Results", f"On the test set ({R['n_test']} tumours), {BEST} achieved the highest ROC-AUC of {f3(b['ROC-AUC'])} (95% CI {f3(b['AUC_lo'])}–{f3(b['AUC_hi'])}), "
  f"with balanced accuracy {f3(b['Balanced accuracy'])}, sensitivity {f3(b['Sensitivity'])} and specificity {f3(b['Specificity'])}; {dl_abs}. "
  f"{de_abs[0].upper() + de_abs[1:]}. The most influential genes in the {BOOST} model were {glist(top_shap[:3])}{' (exploratory, given near-chance discrimination)' if EXPLOR else ''}, and co-expression network analysis identified "
  f"{glist(hub_names[:3])} as hub genes.{kegg_abs}"),
 ("Conclusions", concl_abs),
]
DOC.append(("abstract", ABS))
DOC.append(("kw", "breast cancer; tumour stage; gene expression; TCGA; machine learning; " + ("XGBoost; " if XGB else "gradient boosting; ")
              + ("SHAP; " if SH else "explainable AI; ") + "class imbalance; co-expression network" + ("; KEGG pathway" if KOK else "")))

# ---------- 1 Introduction ----------
H1("INTRODUCTION")
P(f"Breast cancer is the most commonly diagnosed cancer in women worldwide, with about 2.3 million new cases and 666,000 deaths in 2022 {cite('bray')}. "
  "Treatment and prognosis depend strongly on anatomical stage, which the American Joint Committee on Cancer (AJCC) defines from tumour size, regional lymph-node "
  f"involvement and distant metastasis {cite('ajcc')}. Patients with locally advanced (Stage III) or metastatic (Stage IV) disease need more intensive therapy and "
  "have markedly poorer survival than those diagnosed at Stage I–II.")
P(f"Large genomic projects such as The Cancer Genome Atlas (TCGA) have shown that breast tumours are molecularly heterogeneous {cite('tcga')}, and expression-based "
  f"signatures such as PAM50 stratify patients by intrinsic subtype and risk of relapse {cite('parker')}. Machine learning (ML) is now widely used to classify tumours "
  f"from such data {cite('kourou', 'libbrecht', 'tabassum')}, and recent studies integrate multi-omics data with graph neural networks for cancer-type and breast "
  f"cancer subtype classification {cite('alharbi', 'patel')}. Whether the expression profile of the primary tumour also reflects its anatomical extent is less clear; "
  f"network-based studies suggest that changes between breast cancer conditions are often subtle and distributed across interacting genes {cite('wang')}.")
P(f"In this study we built an interpretable, leakage-free ML framework to distinguish early-stage (I–II) from late-stage (III–IV) breast cancer using {R['k']} "
  "cancer-related genes in TCGA-BRCA. Our contributions are: (i) a clearly defined cohort of primary tumours with an explicit staging rule; (ii) a pipeline in "
  "which imputation, scaling and SMOTE are fitted only within training folds, with decision thresholds tuned on out-of-fold predictions and performance reported with "
  "confidence intervals and imbalance-robust metrics; (iii) a comparison of four classifiers with statistical testing of AUC differences; and (iv) biological "
    + (f"interpretation through {INTERP}, differential expression, co-expression network analysis and KEGG pathway enrichment {cite('kegg', 'enrichr')}." if KOK
     else f"interpretation through {INTERP}, differential expression and co-expression network analysis."))

H1("BACKGROUND")
P(f"Gene expression profiling measures the activity of thousands of genes in a tumour and therefore captures molecular processes that are not visible "
  f"in histology {cite('tcga', 'libbrecht')}. Such data pose specific challenges for ML: the number of features is large relative to the number of samples, "
  "which raises the risk of over-fitting; measurements are noisy; many genes are strongly correlated; and clinically important classes are often "
  f"under-represented {cite('kourou')}. Classical models address these problems in different ways. Regularised logistic regression provides an interpretable "
  f"linear baseline, support vector machines with kernels capture non-linear decision boundaries {cite('cortes')}, and tree ensembles such as random forests "
  f"{cite('breiman')} and gradient boosting {cite('xgb') if XGB else cite('friedman')} model interactions between features.")
P("Three methodological pitfalls are common in this setting. First, stage classes are imbalanced, so accuracy is dominated by the majority class and can appear "
  f"high even when a model predicts a single class {cite('saito', 'chicco')}. Second, oversampling with the synthetic minority over-sampling technique (SMOTE) "
  f"{cite('chawla')}, or scaling fitted before the train–test split, leaks information from test samples into training and inflates performance {cite('kaufman', 'blagus')}. "
  "Third, flexible models are hard to interpret, which limits biological insight. SHapley Additive exPlanations (SHAP) address the last problem by attributing each "
  f"prediction to individual features {cite('lundberg2017', 'lundberg2020')}, and interpretable models have been used to find expression biomarkers in early-stage disease {cite('ecmarker')}.")
# ---------- 2 Related work ----------
H1("LITERATURE REVIEW")
P(f"Kourou et al. reviewed ML applications in cancer prognosis and prediction and emphasised the risk of over-fitting when samples are few and features many {cite('kourou')}. "
  f"Ensemble learning combined with feature selection has achieved high accuracy for cancer-type classification from gene expression {cite('tabassum')}, and graph neural "
  f"networks that integrate several omics layers have improved multi-omics cancer classification {cite('alharbi')} and breast cancer subtyping {cite('patel')}. "
  "These tasks separate tissue types or molecular subtypes that differ strongly in expression; classifying stage within a single cancer type is a harder problem "
  "because stage reflects anatomical spread rather than a distinct molecular programme.")
P(f"Wang and Liu identified breast cancer biomarkers through rewiring of gene regulatory networks between conditions {cite('wang')}, and ECMarker used an "
  f"interpretable model to identify gene expression biomarkers that predict clinical outcomes in early disease stages {cite('ecmarker')}. Gene set enrichment "
  f"{cite('gsea')} and KEGG pathway resources {cite('kegg')} are widely used to place candidate genes in a functional context. On the methodological side, Blagus "
  f"and Lusa showed that SMOTE offers limited benefit for high-dimensional data unless it is combined with variable selection {cite('blagus')}, and guidance on "
  f"imbalanced evaluation recommends precision–recall analysis and the Matthews correlation coefficient over accuracy {cite('saito', 'chicco')}. Few studies of "
  "expression-based stage prediction combine leakage-free evaluation, imbalance-robust metrics, model explanation and " + ("network- and pathway-level" if KOK else "network-level") + " analysis; this work "
  "addresses that gap.")

# ---------- 3 Methods ----------
H1("METHODOLOGY")
P("The overall workflow is shown in Figure 1.")
FIG(f"{OUT}/fig1_workflow.png", "Workflow of the study. All preprocessing, oversampling, tuning and threshold selection use training data only; the held-out test set is used once for evaluation.", 0.92)
H2("Data Source")
P(f"Gene expression and clinical data for the TCGA breast invasive carcinoma cohort (TCGA-BRCA) {cite('tcga')} were downloaded from the UCSC Xena browser {cite('xena')} "
  f"as a single sample-by-feature table containing AJCC pathologic stage and normalised RNA-seq expression values (log2 scale, as distributed by Xena) for {len(GENES_ALL)} genes. "
  "The genes were chosen a priori from cancer-related signalling, cell-cycle, DNA-repair, transcriptional-regulation and stem-cell pathways (Supplementary Table S1). "
  "The data are public and de-identified, so no ethical approval was required.")
H2("Cohort and Outcome Definition")
P(f"The table contained {num(R['n_raw'])} samples. Primary solid-tumour samples (TCGA barcode sample type 01; n = {num(R['n_primary'])}) were kept and {other_txt} were removed, "
  f"because normal or metastatic tissue from the same patient carries that patient's stage label but not a primary-tumour profile. " + ("No patient had more than one primary-tumour sample" if R['n_primary'] == R['n_primary_unique'] else "One sample was retained per patient")
  + f" (n = {num(R['n_primary_unique'])}). Tumours were labelled early stage (AJCC Stage I, IA, IB, II, IIA or IIB; class 0) or late stage (Stage III, IIIA, IIIB, IIIC "
  f"or IV; class 1) {cite('ajcc')}. Tumours with unknown (Stage X), discrepant or missing stage were excluded (n = {excl_total}; {excl_txt}). "
  + (f"Genes with more than 20% missing values were removed ({glist(R['genes_dropped_missing'])}). " if R["genes_dropped_missing"] else "No gene had more than 20% missing values. ")
  + f"The remaining missing values ({R['missing_cells_pct']}% of cells) were imputed with the training-set median inside the modelling pipeline.")
H2("Data Splitting, Pre-processing and Class Balancing")
P(f"The cohort was split once into a stratified training set (80%, n = {num(R['n_train'])}) and a held-out test set (20%, n = {R['n_test']}) with a fixed random seed. "
  f"The test set was not used for imputation, scaling, oversampling, tuning or threshold selection. All preprocessing was wrapped in a pipeline in which resampling acts only during fitting, as in imbalanced-learn {cite('imblearn')}, "
  "and re-fitted within each cross-validation fold: median imputation, z-score standardisation, and SMOTE (k = 5 neighbours) to oversample late-stage tumours to parity "
  f"{cite('chawla')}. SMOTE was applied only to training folds; validation folds and the test set kept their natural class ratio. As a sensitivity analysis, every model "
  "was also trained without SMOTE using class weights inversely proportional to class frequency" + (" (scale_pos_weight for XGBoost)." if XGB else "."))
H2("Classification Models and Hyper-parameter Tuning")
P(f"Four classifiers were compared: L2-regularised logistic regression (LR) as a linear baseline {cite('pedregosa')}, a support vector machine with a radial-basis-function "
  f"kernel (SVM) {cite('cortes')}, random forest (RF) {cite('breiman')} and "
  + (f"gradient-boosted trees (XGBoost) {cite('xgb')}. " if XGB else f"gradient-boosted decision trees {cite('friedman')} (scikit-learn HistGradientBoostingClassifier). ")
  + "Hyper-parameters were selected by grid search with "
  "stratified five-fold cross-validation on the training set, maximising ROC-AUC (grids and selected values in Supplementary Table S2).")
H2("Decision Threshold and Performance Evaluation")
P(f"Because the default probability threshold of 0.5 is unsuitable for imbalanced data, each model's threshold was set to maximise Youden's J (sensitivity + specificity − 1) "
  f"{cite('youden')} on out-of-fold predictions for the training set, and then applied unchanged to the test set. Test-set performance was summarised by ROC-AUC, "
  f"precision–recall AUC (average precision) {cite('saito')}, balanced accuracy, sensitivity (recall for late stage), specificity, precision, F1-score and the Matthews "
  f"correlation coefficient (MCC) {cite('chicco')}. Ninety-five per cent confidence intervals were obtained from 1,000 bootstrap resamples of the test set. The AUC of the "
  f"best model was compared with that of each other model using DeLong's test for correlated ROC curves {cite('delong')}. Cross-validated AUC (mean ± SD) on the training "
  "set is also reported to show stability.")
H2("Model Interpretability")
P(("SHAP values were computed for the tuned " + BOOST + " model on the test set using the tree explainer " + cite('lundberg2017', 'lundberg2020') +
   ". Global importance was summarised as the mean absolute SHAP value per gene, and the direction of effect as the correlation between a gene's standardised expression and its SHAP values.")
  if R["shap_method"] == "SHAP" else
  f"Feature importance for the tuned {BOOST} model was estimated by model-agnostic permutation importance {cite('breiman')} on the test set (mean decrease in ROC-AUC over 30 random permutations of each gene), and the direction of effect by the Spearman correlation between a gene's expression and the predicted probability of late stage.")
H2("Differential Expression Analysis")
P(f"Expression of each gene was compared between early- and late-stage tumours in the full cohort with the two-sided Mann–Whitney U test. Effect size was expressed as the "
  f"rank-biserial correlation r (positive when expression is higher in late stage) and as the difference in mean log2 expression. P-values were adjusted with the "
  f"Benjamini–Hochberg procedure {cite('bh')}, and genes with a false discovery rate (FDR) below 0.05 were considered stage-associated.")
H2("Gene Co-expression Network and Hub Gene Identification")
P(f"Pairwise Pearson correlations between all {R['k']} genes were computed on training samples. Undirected co-expression networks were built in NetworkX {cite('networkx')} by "
  "connecting gene pairs with |r| ≥ 0.7 (strong), |r| ≥ 0.5 (moderate) and |r| ≥ 0.3 (weak). Hub genes were ranked by degree in the network at |r| ≥ 0.5, or at |r| ≥ 0.3 "
  f"if the 0.5 network had fewer than ten edges. Correlation networks describe co-expression and do not establish regulatory direction {cite('wgcna')}.")
if KOK:
  H2("KEGG Pathway Enrichment Analysis")
  P(f"The union of the 20 most important genes from model interpretation and all stage-associated genes (FDR &lt; 0.05), up to 30 genes, was tested for over-representation "
  f"in KEGG pathways {cite('kegg')} (KEGG_2021_Human library) with Enrichr {cite('enrichr')} through GSEApy {cite('gseapy')}. Pathways with a Benjamini–Hochberg adjusted "
  "p-value below 0.05 were considered enriched.")
H2("Software and Reproducibility")
_libs = [f"scikit-learn {V['scikit-learn']}"] + ([f"imbalanced-learn {V['imbalanced-learn']}"] if V.get("imbalanced-learn") else []) \
        + ([f"XGBoost {V['xgboost']}"] if V.get("xgboost") else []) + [f"pandas {V['pandas']}", f"SciPy", f"NetworkX {V['networkx']}"] \
        + (["SHAP"] if SH else []) + (["GSEApy"] if KOK else [])
P(f"Analyses were run in Python {V['python']} with {plist(_libs)}"
  + ("" if V.get("imbalanced-learn") else "; SMOTE was implemented as described by Chawla et al. " + cite("chawla"))
  + f". The random seed was fixed at {SEED} throughout.")

# ---------- 4 Results ----------
H1("RESULTS AND DISCUSSION")
H2("Experimental Setup")
P(f"After filtering, {num(R['n'])} primary tumours were analysed: {num(R['n_early'])} early-stage and {R['n_late']} late-stage ({R['pct_late']}% late; Table 1, Figure 2). "
  f"The stratified split gave {num(R['n_train'])} training tumours ({R['n_train_early']} early, {R['n_train_late']} late) and {R['n_test']} test tumours "
  f"({R['n_test_early']} early, {R['n_test_late']} late). Within each training fold SMOTE balanced the classes; applied to the full training set it would give "
  f"{num(2 * R['n_train_early'])} samples ({R['n_train_early']} per class). Because of the imbalance, a trivial classifier that labels every tumour as early would "
  f"reach {R['acc_all_early_test']}% accuracy on the test set; accuracy alone is therefore not informative here.")
t1 = [["Source", "TCGA-BRCA via UCSC Xena"], ["Samples in downloaded table", num(R["n_raw"])],
      ["Primary tumours (sample type 01)", num(R["n_primary"])], ["One sample per patient", num(R["n_primary_unique"])],
      ["Excluded: stage X, discrepant or missing", str(excl_total)], ["Analysed tumours", num(R["n"])],
      ["Early stage, I–II (class 0)", f"{num(R['n_early'])} ({100 - R['pct_late']:.1f}%)"], ["Late stage, III–IV (class 1)", f"{R['n_late']} ({R['pct_late']}%)"],
      ["Genes in panel", str(R["k"])], ["Training set (early / late)", f"{num(R['n_train'])} ({R['n_train_early']} / {R['n_train_late']})"],
      ["Test set (early / late)", f"{R['n_test']} ({R['n_test_early']} / {R['n_test_late']})"],
      ["Training set after SMOTE (per class)", f"{num(2 * R['n_train_early'])} ({R['n_train_early']})"]]
TAB(tcap("Cohort characteristics and data partition."), ["Characteristic", "Value"], t1, [0.62, 0.38])
FIG(f"{OUT}/fig2_stage_distribution.png", "Distribution of AJCC pathologic stage among the analysed primary tumours. Blue, early stage (I–II); orange, late stage (III–IV).", 0.8)

H2("Model Performance")
cv_txt = ("Cross-validated AUC on the training set was close to the test AUC for every model" if all(abs(RES.loc[m, "ROC-AUC"] - RES.loc[m, "CV_AUC_mean"]) < 0.05 for m in RES.index)
          else "Cross-validated and test AUCs differed by more than 0.05 for at least one model, reflecting the variance of estimates from a test set of this size")
P(f"Table 2 summarises test-set performance at each model's tuned threshold. {BEST} achieved the highest ROC-AUC, {f3(b['ROC-AUC'])} (95% CI {f3(b['AUC_lo'])}–{f3(b['AUC_hi'])}), "
  f"indicating {word} discrimination between early- and late-stage tumours; the other models reached {f3(auc_range_other[0])}–{f3(auc_range_other[1])}. "
  + (f"The 95% confidence interval of the AUC excluded 0.5 for {plist(above_chance)}. " if above_chance else "No model's AUC confidence interval excluded 0.5. ")
  + (f"DeLong tests comparing {BEST} with each other model gave " + plist([f"{pf(dl[m])} ({m})" for m in others]) + ". "
     + ("The differences between models were therefore not statistically significant. " if not dl_sig else "")))
P(f"At its tuned threshold of {b['Threshold']:.2f}, {BEST} identified {int(b['TP'])} of {R['n_test_late']} late-stage test tumours (sensitivity {f3(b['Sensitivity'])}) "
  f"and correctly classified {int(b['TN'])} of {R['n_test_early']} early-stage tumours (specificity {f3(b['Specificity'])}; Figure 4), with balanced accuracy "
  f"{f3(b['Balanced accuracy'])} (95% CI {f3(b['BA_lo'])}–{f3(b['BA_hi'])}) and MCC {f3(b['MCC'])}. Its precision–recall AUC was {f3(b['PR-AUC'])}, against a "
  f"no-skill baseline equal to the late-stage prevalence of {R['n_test_late'] / R['n_test']:.3f} (Figure 3b). {cv_txt}.")
t2 = []
for m in RES.sort_values("ROC-AUC", ascending=False).index:
    r_ = RES.loc[m]
    t2.append([m, f"{f3(r_['ROC-AUC'])} ({f3(r_['AUC_lo'])}–{f3(r_['AUC_hi'])})", f3(r_["PR-AUC"]),
               f"{f3(r_['Balanced accuracy'])} ({f3(r_['BA_lo'])}–{f3(r_['BA_hi'])})", f3(r_["Sensitivity"]), f3(r_["Specificity"]),
               f3(r_["F1"]), f3(r_["MCC"]), f"{f3(r_['CV_AUC_mean'])} ± {f3(r_['CV_AUC_sd'])}"])
TAB(tcap("Performance on the held-out test set (late stage = positive class). Bal. acc., balanced accuracy; Sens., sensitivity; Spec., specificity. Values in brackets are 95% bootstrap confidence intervals. CV AUC, five-fold cross-validated ROC-AUC on the training set (mean ± SD)."),
    ["Model", "ROC-AUC (95% CI)", "PR-AUC", "Bal. acc. (95% CI)", "Sens.", "Spec.", "F1", "MCC", "CV AUC"],
    t2, [0.14, 0.17, 0.09, 0.17, 0.07, 0.07, 0.07, 0.07, 0.15], small=True)
FIG(f"{OUT}/fig3_roc_pr.png", "(a) ROC curves and (b) precision–recall curves of the four classifiers on the held-out test set. Dashed lines show chance performance.", 1.0)
FIG(f"{OUT}/fig4_confusion_matrix.png", f"Confusion matrix of {BEST} on the held-out test set at the threshold selected on out-of-fold training predictions. Percentages are row-wise.", 0.45)
P(f"In the sensitivity analysis without SMOTE, class-weighted models gave test AUCs that differed from the SMOTE-based models by at most {cw_delta:.3f}, with {cw_best} performing best "
  f"(AUC {f3(RES_CW.loc[cw_best, 'ROC-AUC'])}; Supplementary Table S3). "
  + ("Conclusions were therefore not sensitive to the choice of imbalance strategy." if cw_delta < 0.05
     else "Performance therefore depended to some extent on the imbalance strategy, which should be considered when interpreting small differences between models."))

H2("Model Interpretability")
imp_word = "mean absolute SHAP value" if R["shap_method"] == "SHAP" else "permutation importance"
P(f"Figure 5 shows the genes with the largest influence on the {BOOST} model (test AUC {f3(RES.loc[BOOST, 'ROC-AUC'])}). Ranked by {imp_word}, the five most important genes were "
  f"{glist(top_shap[:5])}. "
  + (f"Higher expression of {glist(late_genes)} pushed predictions towards late stage" if late_genes else "")
  + ("; " if late_genes and early_genes else "")
  + ((f"higher expression of {glist(early_genes)} favoured early stage" if late_genes else f"Higher expression of {glist(early_genes)} favoured early stage") if early_genes else "")
  + ". These values describe how the model uses each gene; they are not tests of statistical significance, which are reported in the next section." + imp_caveat)
FIG(f"{OUT}/fig5_shap.png", (f"SHAP summary plot for the {BOOST} model on the held-out test set. Each point is a tumour; its position shows the gene's contribution to the predicted "
     "log-odds of late stage, and its colour the standardised expression (red, high; blue, low). Genes are ordered by mean absolute SHAP value.") if R["shap_method"] == "SHAP"
    else f"Permutation importance of the 15 most important genes for the {BOOST} model on the held-out test set (mean decrease in ROC-AUC). Orange, higher expression associated with a higher predicted probability of late stage; blue, with early stage.", 0.78)

H2("Stage-associated Gene Expression")
if R["n_sig"]:
    P(f"Of the {R['k']} genes, {R['n_sig']} differed between early- and late-stage tumours at FDR &lt; 0.05 (Table 3; full results in Supplementary Table S4). "
      + (f"Expression was higher in late-stage tumours for {glist(R['sig_up'][:8])}" if R["sig_up"] else "")
      + ("; " if R["sig_up"] and R["sig_down"] else "")
      + (f"it was lower for {glist(R['sig_down'][:8])}" if R["sig_down"] else "")
      + f". Effect sizes were small (largest |r| = {DE['Effect size r'].abs().max():.2f}), consistent with the modest discrimination of the classifiers. "
      + (f"Of the ten genes most important to the {BOOST} model, {glist(overlap_shap_de)} {'was' if len(overlap_shap_de) == 1 else 'were'} also significant in this univariate analysis." if overlap_shap_de
         else f"None of the ten genes most important to the {BOOST} model reached FDR &lt; 0.05 individually, suggesting that the model draws on combinations of weak signals."))
else:
    P(f"No gene differed between early- and late-stage tumours after FDR correction (smallest FDR = {DE['FDR'].min():.3f}; Table 3, Supplementary Table S4). "
      f"The largest effect size was |r| = {DE['Effect size r'].abs().max():.2f}, indicating that stage-related differences in these genes are small.")
t3 = [[it(r_.Gene), f"{r_['Mean early']:.2f}".replace("-", "−"), f"{r_['Mean late']:.2f}".replace("-", "−"), f"{r_['Difference (late − early)']:+.2f}".replace("-", "−"), f"{r_['Effect size r']:+.2f}".replace("-", "−"),
       ("&lt; 0.001" if r_.p < 0.001 else f"{r_.p:.3f}"), qf(r_.FDR)] for _, r_ in DE.head(10).iterrows()]
TAB(tcap("Ten genes with the smallest p-values for differential expression between early- and late-stage tumours (Mann–Whitney U test). Means are on the log2 expression scale; r is the rank-biserial correlation (positive, higher in late stage); FDR, Benjamini–Hochberg adjusted p-value."),
    ["Gene", "Mean early", "Mean late", "Difference", "r", "p", "FDR"], t3, [0.16, 0.14, 0.14, 0.14, 0.12, 0.15, 0.15])
FIG(f"{OUT}/fig6_heatmap.png", "Expression (z-score) of the 20 genes with the smallest differential-expression p-values across all analysed tumours, ordered by stage group. The vertical line separates early- from late-stage tumours.", 0.95)

H2("Gene Co-expression Network and Hub Genes")
n07, n05, n03 = R["net"]["0.7"], R["net"]["0.5"], R["net"]["0.3"]
e07 = ("; the strong edges were " + plist([f"{it(a_)}–{it(b_)} (r = {r_:.2f})" for a_, b_, r_ in R["edges07"][:6]])) if 0 < n07["edges"] <= 6 else ""
P(f"Correlations among the panel genes are shown in Figure 7. At |r| ≥ 0.7 the network contained {n07['edges']} edge{'s' if n07['edges'] != 1 else ''} among {n07['nodes']} genes{e07}. "
  f"Lowering the threshold to 0.5 and 0.3 gave {n05['edges']} edges ({n05['nodes']} genes) and {n03['edges']} edges ({n03['nodes']} genes), respectively. "
  f"Hub genes were ranked in the |r| ≥ {R['hub_thr']} network (Figure 8): the most connected were "
  + plist([f"{it(g)} (degree {k_})" for g, k_ in hubs[:5]]) + ". "
  + (f"Among the top ten hubs, {glist(overlap_hub)} {'was' if len(overlap_hub) == 1 else 'were'} also important to the classifier or significant in the differential-expression analysis." if overlap_hub
     else "The top hubs did not overlap with the genes most important to the classifier or with the stage-associated genes, indicating that co-expression centrality and stage association capture different properties."))
FIG(f"{OUT}/fig7_correlation.png", "Pearson correlation matrix of the panel genes in the training set, ordered by hierarchical clustering (average linkage).", 0.85)
FIG(f"{OUT}/fig8_network_hubs.png", f"(a) Gene co-expression network at |r| ≥ {R['hub_thr']} in the training set; orange edges denote positive and blue edges negative correlations, node size is proportional to degree, and the five top hubs are shaded orange. (b) Degree of the ten most connected genes.", 1.0)

if KOK:
    H2("KEGG Pathway Enrichment Analysis")
    kt = R["kegg_top"]
    P(f"Of the {len(R['kegg_input'])} genes tested, enrichment analysis identified {R['kegg_n_sig']} KEGG pathways at adjusted p &lt; 0.05 (Table 4, Figure 9). "
      f"The most enriched were {plist([f'{t} (adjusted {pf(q)})' for t, _, q, _ in kt[:3]])}. "
      "Because the panel was selected from cancer-related pathways, enrichment of cancer pathways is expected by design; the analysis therefore places the "
      "stage-associated genes in a functional context rather than providing independent validation of the classifier.")
    t4 = [[t, ov, (f"{q:.1e}" if q < 0.001 else f"{q:.3f}"), ", ".join(it(g) for g in str(gs).split(";")[:8])] for t, ov, q, gs in kt]
    TAB(tcap("Top KEGG pathways enriched among the genes ranked by the model and the differential-expression analysis (Enrichr, KEGG_2021_Human). Overlap, input genes in pathway / pathway size."),
        ["KEGG pathway", "Overlap", "Adjusted p", "Genes"], t4, [0.33, 0.1, 0.12, 0.45], small=True)
    FIG(f"{OUT}/fig9_kegg.png", "Top KEGG pathways ranked by Benjamini–Hochberg adjusted p-value. The dashed line marks FDR = 0.05; numbers give the overlap.", 0.9)

# ---------- 5 Discussion ----------
H2("Discussion")
P(f"Using a leakage-free pipeline on {num(R['n'])} TCGA-BRCA primary tumours, we found that a panel of {R['k']} cancer-related genes "
  + (f"has only {word} ability to discriminate early- from late-stage breast cancer" if word in ("poor", "modest") else f"discriminates early- from late-stage breast cancer with {word} performance")
  + f": the best model, {BEST}, reached a test ROC-AUC of {f3(b['ROC-AUC'])} (95% CI {f3(b['AUC_lo'])}–{f3(b['AUC_hi'])}) and a balanced "
  f"accuracy of {f3(b['Balanced accuracy'])}. "
  + ("Linear and non-linear models performed similarly, so there is no evidence that complex non-linear interactions among these genes add predictive information. "
     if not dl_sig else f"{BEST} outperformed {plist(dl_sig)} significantly. ")
  + "These estimates are lower than the accuracies often reported for expression-based cancer classification, but they answer a harder question.")
P("Stage reflects the anatomical extent of disease — tumour size, nodal spread and metastasis — and the time at which the tumour was detected. These properties are only "
  "partly encoded in the expression profile of the primary tumour, whereas molecular subtype and proliferation, which expression captures well, are related to but distinct "
  f"from stage {cite('tcga', 'parker')}. Our results also illustrate why accuracy is a poor guide in imbalanced problems {cite('saito', 'chicco')}: because "
  f"{100 - R['pct_late']:.0f}% of tumours are early stage, a model that predicts early stage for every tumour attains {100 - R['pct_late']:.0f}% accuracy while having no discriminative value. "
  "Reporting ROC-AUC with confidence intervals, balanced accuracy, sensitivity, specificity and MCC, and fitting SMOTE and scaling only within training folds "
  f"{cite('kaufman', 'blagus')}, gives an honest picture of what the data support.")
focus = list(dict.fromkeys(top_shap[:3] + sig[:3] + hub_names[:3]))
notes = [f"{it(g)} encodes {GENE_INFO[g]}" for g in focus if g in GENE_INFO][:6]
if notes:
    P("Several of the genes highlighted by the model, the differential-expression analysis or the network have known roles in cancer biology. "
      + "; ".join(notes) + "." + imp_caveat + " Their association with stage in this cohort is correlative: the data cannot show whether these genes drive progression or reflect "
      "differences in molecular subtype or tumour composition between stage groups.")
P(("The co-expression network and pathway analyses add context but should be interpreted cautiously. " if KOK else "The co-expression network adds context but should be interpreted cautiously. ")
  + f"Correlation-based networks describe co-expression, not regulation {cite('wgcna')}, and hub status in a network built from a pre-selected panel partly reflects how many related genes were included. "
  + ("Likewise, enrichment of cancer pathways is expected when the input genes were chosen from such pathways. " if KOK else "")
  + (f"Genes highlighted by more than one analysis ({glist(CONV[:6])}) are the most credible candidates for follow-up." if CONV
     else "No gene was highlighted consistently by model importance, univariate stage association and network centrality, which again indicates that the stage signal in this panel is weak and distributed across many genes."))
H2("Limitations and Future Work")
P("This study has limitations. First, it uses a single cohort without external validation; independent cohorts are needed to confirm generalisability. Second, the gene "
  "panel was fixed in advance, which protects against selection leakage but limits discovery; whole-transcriptome feature selection nested inside cross-validation may "
  f"capture additional signal. Third, the test set contained only {R['n_test_late']} late-stage tumours, so performance estimates have wide confidence intervals. Fourth, TCGA "
  "samples were collected over many years under different editions of the AJCC staging system, and clinical covariates such as molecular subtype, grade and age were not "
  "included. Finally, SMOTE creates synthetic samples by interpolation, which may not reflect real biological variation.")
P("Future work should validate the framework in independent cohorts, combine expression with clinicopathological variables and other omics layers, and test whether "
  "stage-associated genes add prognostic information beyond stage itself.")

# ---------- 6 Conclusion ----------
H1("CONCLUSION")
P(f"We present an interpretable and leakage-free machine-learning framework for classifying breast cancer stage from gene expression. On {num(R['n'])} TCGA-BRCA primary "
  f"tumours, the best model ({BEST}) achieved a test ROC-AUC of {f3(b['ROC-AUC'])}"
  + (f", and {glist(CONV[:5])} {'was' if len(CONV) == 1 else 'were'} highlighted by more than one of the {INTERP}, differential-expression and network analyses. " if CONV
     else f"; no gene was consistently highlighted across the {INTERP}, differential-expression and network analyses. ") + concl_abs)

# ---------- Declarations ----------
H1("DECLARATIONS")
P("<b>Data availability.</b> The TCGA-BRCA data analysed in this study are publicly available from the UCSC Xena browser (https://xenabrowser.net).")
P(f"<b>Code availability.</b> {CODE_AVAILABILITY}")
P(f"<b>Funding.</b> {FUNDING}")
P("<b>Conflict of interest.</b> The authors declare no potential conflicts of interest with respect to the research, authorship and/or publication of this article.")
P("<b>Ethics statement.</b> This study used only publicly available, de-identified data and did not require ethical approval.")
P("<b>Acknowledgements.</b> The results shown here are based upon data generated by the TCGA Research Network: https://www.cancer.gov/tcga.")

DOC.append(("refs", [REFS[k] for k in _ORDER]))

# ---------- Supplementary ----------
DOC.append(("pagebreak",))
H1("SUPPLEMENTARY MATERIAL")
rows_s1 = [[it(g) for g in GENES[i:i + 8]] + [""] * (8 - len(GENES[i:i + 8])) for i in range(0, len(GENES), 8)]
TAB(f"Table S1: The {R['k']} genes analysed.", [""] * 8, rows_s1, [0.125] * 8, small=True)
rows_s2 = [[m, "; ".join(f"{k}: {v}" for k, v in PARAMS[m]["grid"].items()), "; ".join(f"{k} = {v}" for k, v in PARAMS[m]["best"].items())] for m in PARAMS]
TAB("Table S2: Hyper-parameter grids searched with stratified five-fold cross-validation and the selected values.", ["Model", "Grid", "Selected"], rows_s2, [0.18, 0.5, 0.32], small=True)
rows_s3 = [[m, f3(RES_CW.loc[m, "ROC-AUC"]), f3(RES_CW.loc[m, "PR-AUC"]), f3(RES_CW.loc[m, "Balanced accuracy"]), f3(RES_CW.loc[m, "Sensitivity"]),
            f3(RES_CW.loc[m, "Specificity"]), f3(RES_CW.loc[m, "MCC"])] for m in RES_CW.sort_values("ROC-AUC", ascending=False).index]
TAB("Table S3: Sensitivity analysis: test-set performance of class-weighted models trained without SMOTE.",
    ["Model", "ROC-AUC", "PR-AUC", "Balanced accuracy", "Sensitivity", "Specificity", "MCC"], rows_s3, [0.22, 0.12, 0.12, 0.16, 0.13, 0.13, 0.12], small=True)
P("<b>Table S4</b> (full differential-expression results)" + (" and <b>Table S5</b> (full KEGG enrichment results) are" if KOK else " is") + " provided as a spreadsheet file with this article.")
print("Paper text assembled:", len(DOC), "blocks;", len(_ORDER), "references cited;", FIGN[0], "figures;", TABN[0], "main tables")

In [ ]:
# ===================== 12. Render Word (in the style of your template) and PDF =====================
from docx import Document
from docx.shared import Pt, Inches, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH as AL
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml.ns import qn
from docx.oxml import OxmlElement
TNR = "Times New Roman"

def _runs(par, txt, size=12, bold=False, italic=False):
    st = {"i": italic, "b": bold, "sup": False}
    for tok in re.split(r"(<[^>]+>)", txt.replace("&nbsp;", " ")):
        if not tok: continue
        if tok.startswith("<"):
            tg = tok.strip("<>/ ").lower()
            if tg in st: st[tg] = not tok.startswith("</") or (tg == "b" and bold) or (tg == "i" and italic)
            continue
        r = par.add_run(_html.unescape(tok))
        r.font.name = TNR; r.font.size = Pt(size); r.font.color.rgb = RGBColor(0, 0, 0)
        rf = r._element.rPr.find(qn("w:rFonts"))
        for k in ("w:ascii", "w:hAnsi", "w:cs", "w:eastAsia"): rf.set(qn(k), TNR)
        r.bold = st["b"]; r.italic = st["i"]; r.font.superscript = st["sup"]

def render_docx(path):
    use_tpl = bool(TEMPLATE_DOCX) and os.path.exists(TEMPLATE_DOCX)
    D = Document(TEMPLATE_DOCX) if use_tpl else Document()
    body = D.element.body
    for el in list(body):
        if el.tag != qn("w:sectPr"): body.remove(el)
    sec = D.sections[0]
    sec.page_width, sec.page_height = Inches(8.5), Inches(11)
    sec.left_margin = sec.right_margin = sec.top_margin = sec.bottom_margin = Inches(1)
    def style_or_none(name):
        try: return D.styles[name]
        except KeyError: return None
    def para(txt, size=12, align=AL.JUSTIFY, bold=False, after=12, before=0, style=None, keep=False, hang=None):
        p = D.add_paragraph(style=style_or_none(style) if style else None)
        pf = p.paragraph_format; pf.space_after = Pt(after); pf.space_before = Pt(before); pf.line_spacing = 1.0
        p.alignment = align; pf.keep_with_next = keep; pf.widow_control = True
        if hang: pf.left_indent = Inches(hang); pf.first_line_indent = -Inches(hang)
        _runs(p, txt, size, bold); return p
    def shade(cell, hex_):
        tcPr = cell._element.get_or_add_tcPr(); sh = OxmlElement("w:shd")
        sh.set(qn("w:val"), "clear"); sh.set(qn("w:color"), "auto"); sh.set(qn("w:fill"), hex_); tcPr.append(sh)
    _brk = [False]
    if DEMO: para("DEMO RUN ON SYNTHETIC DATA - NOT REAL RESULTS", 12, AL.CENTER, True)
    for blk in DOC:
        k = blk[0]
        if k == "title": para(blk[1], 18, AL.CENTER, True, after=12, style="Title")
        elif k == "authors": para(blk[1], 12, AL.CENTER, True, after=12)
        elif k == "affil":
            if blk[1].startswith("*"):
                em = re.search(r"[\w.+-]+@[\w.-]+", blk[1]).group(0)
                para(f"<b>*Corresponding author’s E-mail:</b> {em}", 12, AL.CENTER, False, after=24, before=12)
            else: para(blk[1], 12, AL.CENTER, False, after=0)
        elif k == "h1":
            hp = para(blk[1].upper(), 12, AL.LEFT, True, after=12, before=12, style="Heading 1", keep=True)
            if _brk[0]: hp.paragraph_format.page_break_before = True; _brk[0] = False
        elif k == "h2": para(blk[1], 12, AL.LEFT, True, after=6, before=6, keep=True)
        elif k == "p": para(blk[1])
        elif k == "abstract":
            para("ABSTRACT", 12, AL.JUSTIFY, True, after=12, before=12, keep=True)
            para(" ".join(t for _, t in blk[1]))
        elif k == "kw": para(f"<b>Keywords:</b> {blk[1]}", after=18)
        elif k == "table":
            _, cap, head, rows, widths, note, small = blk
            para(cap, 12, AL.LEFT, True, after=6, before=6, keep=True)
            allrows = ([head] if any(head) else []) + rows
            ncol = len(allrows[0]); widths = widths or [1 / ncol] * ncol
            t = D.add_table(rows=len(allrows), cols=ncol); t.alignment = WD_TABLE_ALIGNMENT.CENTER; t.autofit = False
            tblPr = t._tbl.tblPr
            bd = OxmlElement("w:tblBorders")
            for side in ("top", "left", "bottom", "right", "insideH", "insideV"):
                e = OxmlElement(f"w:{side}"); e.set(qn("w:val"), "single"); e.set(qn("w:sz"), "4"); e.set(qn("w:space"), "0"); e.set(qn("w:color"), "000000"); bd.append(e)
            tblPr.append(bd)
            tw = OxmlElement("w:tblW"); tw.set(qn("w:w"), str(int(9360 * sum(widths)))); tw.set(qn("w:type"), "dxa"); tblPr.append(tw)
            lay = OxmlElement("w:tblLayout"); lay.set(qn("w:type"), "fixed"); tblPr.append(lay)
            for j, col in enumerate(t.columns): col.width = Inches(6.5 * widths[j])
            for i, r_ in enumerate(allrows):
                for j, c in enumerate(r_):
                    cell = t.cell(i, j); cell.width = Inches(6.5 * widths[j])
                    cp = cell.paragraphs[0]; cp.paragraph_format.space_after = Pt(0); cp.alignment = AL.LEFT
                    _runs(cp, str(c), 9 if small else 10, bold=(i == 0 and any(head)))
                    if i == 0 and any(head): shade(cell, "E7E6E6")
            para("", after=6)
        elif k == "fig":
            _, path_, cap, w = blk
            p = D.add_paragraph(); p.alignment = AL.CENTER; p.paragraph_format.keep_with_next = True
            p.add_run().add_picture(path_, width=Inches(min(6.5, 6.3 * w)))
            para(cap, 11, AL.CENTER, True, after=14)
        elif k == "refs":
            para("REFERENCES", 12, AL.LEFT, True, after=12, before=12, style="Heading 1", keep=True)
            for i, rr in enumerate(blk[1], 1): para(f"{i}.\t{rr}", 12, AL.JUSTIFY, after=4, hang=0.35)
        elif k == "pagebreak": _brk[0] = True
    D.save(path)

# PDF: converted from the Word file with LibreOffice when available (identical layout); otherwise drawn with ReportLab in the same style
from reportlab.lib.pagesizes import letter
from reportlab.lib.units import inch
from reportlab.lib import colors
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.enums import TA_JUSTIFY, TA_CENTER, TA_LEFT
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, KeepTogether, PageBreak
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.lib.fonts import addMapping

def _register_fonts():
    lib = "/usr/share/fonts/truetype/liberation/"
    mp = os.path.join(os.path.dirname(matplotlib.__file__), "mpl-data", "fonts", "ttf")
    for c in [tuple(lib + f for f in ["LiberationSerif-Regular.ttf", "LiberationSerif-Bold.ttf", "LiberationSerif-Italic.ttf", "LiberationSerif-BoldItalic.ttf"]),
              tuple(os.path.join(mp, f) for f in ["DejaVuSerif.ttf", "DejaVuSerif-Bold.ttf", "DejaVuSerif-Italic.ttf", "DejaVuSerif-BoldItalic.ttf"])]:
        if all(os.path.exists(p) for p in c):
            for nm, p in zip(["Serif", "Serif-B", "Serif-I", "Serif-BI"], c): pdfmetrics.registerFont(TTFont(nm, p))
            addMapping("Serif", 0, 0, "Serif"); addMapping("Serif", 1, 0, "Serif-B"); addMapping("Serif", 0, 1, "Serif-I"); addMapping("Serif", 1, 1, "Serif-BI")
            return "Serif", "Serif-B"
    return "Times-Roman", "Times-Bold"

def render_pdf_reportlab(path):
    FONT, FB = _register_fonts()
    S = lambda n, **kw: ParagraphStyle(n, fontName=FONT, fontSize=12, leading=14, alignment=TA_JUSTIFY, spaceAfter=12, **kw)
    ss = {"title": S("t", spaceBefore=0).clone("t", fontName=FB, fontSize=18, leading=22, alignment=TA_CENTER),
          "authors": S("a").clone("a", fontName=FB, alignment=TA_CENTER), "affil": S("af").clone("af", alignment=TA_CENTER, spaceAfter=0),
          "h1": S("h1").clone("h1", fontName=FB, alignment=TA_LEFT, spaceBefore=12, keepWithNext=1),
          "h2": S("h2").clone("h2", fontName=FB, alignment=TA_LEFT, spaceBefore=6, spaceAfter=6, keepWithNext=1),
          "p": S("p"), "cap": S("c").clone("c", fontName=FB, fontSize=11, leading=13, alignment=TA_CENTER, spaceBefore=4, spaceAfter=14),
          "tcap": S("tc").clone("tc", fontName=FB, alignment=TA_LEFT, spaceAfter=6),
          "cell": S("ce").clone("ce", fontSize=10, leading=12, alignment=TA_LEFT, spaceAfter=0),
          "cells": S("cs").clone("cs", fontSize=9, leading=11, alignment=TA_LEFT, spaceAfter=0),
          "ref": S("r").clone("r", leftIndent=25, firstLineIndent=-25, spaceAfter=4)}
    W = letter[0] - 2 * inch; story = []
    def page(c, d):
        c.saveState(); c.setFont(FONT, 10); c.drawCentredString(letter[0] / 2, 0.5 * inch, str(d.page))
        if DEMO:
            c.setFillColor(colors.Color(0.85, 0.1, 0.1, alpha=0.18)); c.setFont(FB, 48); c.translate(letter[0] / 2, letter[1] / 2); c.rotate(45)
            c.drawCentredString(0, 30, "DEMO - SYNTHETIC DATA"); c.drawCentredString(0, -30, "NOT REAL RESULTS")
        c.restoreState()
    for blk in DOC:
        k = blk[0]
        if k in ("title", "authors", "p"): story.append(Paragraph(blk[1], ss[k]))
        elif k == "affil":
            if blk[1].startswith("*"):
                em = re.search(r"[\w.+-]+@[\w.-]+", blk[1]).group(0)
                story.append(Spacer(1, 12)); story.append(Paragraph(f"<b>*Corresponding author’s E-mail:</b> {em}", ss["affil"])); story.append(Spacer(1, 24))
            else: story.append(Paragraph(blk[1], ss["affil"]))
        elif k == "h1": story.append(Paragraph(blk[1].upper(), ss["h1"]))
        elif k == "h2": story.append(Paragraph(blk[1], ss["h2"]))
        elif k == "abstract":
            story.append(Paragraph("ABSTRACT", ss["h1"])); story.append(Paragraph(" ".join(t for _, t in blk[1]), ss["p"]))
        elif k == "kw": story.append(Paragraph(f"<b>Keywords:</b> {blk[1]}", ss["p"]))
        elif k == "table":
            _, cap, head, rows, widths, note, small = blk
            st = ss["cells"] if small else ss["cell"]
            data = ([[Paragraph(f"<b>{h}</b>", st) for h in head]] if any(head) else []) + [[Paragraph(str(c), st) for c in r_] for r_ in rows]
            widths = widths or [1 / len(rows[0])] * len(rows[0])
            t = Table(data, colWidths=[w_ * W for w_ in widths], repeatRows=1 if any(head) else 0)
            sty = [("GRID", (0, 0), (-1, -1), 0.5, colors.black), ("VALIGN", (0, 0), (-1, -1), "TOP"),
                   ("TOPPADDING", (0, 0), (-1, -1), 2), ("BOTTOMPADDING", (0, 0), (-1, -1), 2)]
            if any(head): sty.append(("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#E7E6E6")))
            t.setStyle(TableStyle(sty)); story.append(KeepTogether([Paragraph(cap, ss["tcap"]), t, Spacer(1, 14)]))
        elif k == "fig":
            _, path_, cap, w = blk
            from reportlab.lib.utils import ImageReader
            iw, ih = ImageReader(path_).getSize(); wd = min(W, 6.3 * inch * w); ht = wd * ih / iw
            if ht > 7.5 * inch: ht = 7.5 * inch; wd = ht * iw / ih
            story.append(KeepTogether([Image(path_, width=wd, height=ht), Paragraph(cap, ss["cap"])]))
        elif k == "refs":
            story.append(Paragraph("REFERENCES", ss["h1"]))
            for i, rr in enumerate(blk[1], 1): story.append(Paragraph(f"{i}.&nbsp;&nbsp;{rr}", ss["ref"]))
        elif k == "pagebreak": story.append(PageBreak())
    SimpleDocTemplate(path, pagesize=letter, leftMargin=inch, rightMargin=inch, topMargin=inch, bottomMargin=inch,
                      title=re.sub("<.*?>", "", TITLE)).build(story, onFirstPage=page, onLaterPages=page)

def render_pdf(path, docx_path):
    import shutil, subprocess
    soff = shutil.which("soffice") or shutil.which("libreoffice")
    if soff:
        try:
            subprocess.run([soff, "--headless", "--convert-to", "pdf", "--outdir", os.path.dirname(os.path.abspath(path)) or ".", docx_path],
                           check=True, capture_output=True, timeout=300)
            produced = os.path.splitext(os.path.abspath(docx_path))[0] + ".pdf"
            if produced != os.path.abspath(path): shutil.move(produced, path)
            return "LibreOffice"
        except Exception as e:
            print("LibreOffice conversion failed, using ReportLab:", e)
    render_pdf_reportlab(path); return "ReportLab"

render_docx("paper.docx"); print("PDF via", render_pdf("paper.pdf", "paper.docx"))

import shutil
shutil.make_archive("results", "zip", OUT)
print("Written: paper.pdf, paper.docx, results.zip")

In [ ]:
from google.colab import files
for f in ["paper.pdf", "paper.docx", "results.zip"]:
    files.download(f)